# RNN-1 · 순서 데이터 · RNN · LSTM · GRU

**한 줄 요약**

- 순서가 답을 정하는 데이터에서 합
- 평균 요약이 왜 실패하는지 확인하고, 요약을 칸마다 넘기는 RNN과 문이 달린 LSTM · GRU를 손계산 · 코드
- 실제 센서 데이터(UCI HAR)로 비교함

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 합 · 평균 요약이 '3이 7보다 먼저인가'에서 찍기 수준(0.4880)이 되는 이유와 마지막 값이 60% 근처인 이유를 숫자로 설명한다
2. `nn.RNNCell` + `for` 문으로 hidden state를 손으로 펼쳐 `nn.RNN`과 같음을 보이고, 길이가 달라도 파라미터 수가 같은 이유를 말한다
3. 층 · 방향에 따른 `out` · `h_n` 모양을 손으로 쓰고, 길이가 다른 줄에서는 `lens - 1`로 진짜 끝을 꺼낸다
4. 칸별 기울기 크기로 기울기 소실을 보이고, LSTM forget 문의 역할과 RNN · GRU · LSTM 파라미터(1 · 3 · 4벌)를 손계산한다
5. UCI HAR에서 세 모델을 같은 조건 · 파라미터 맞춤 · 여러 seed로 비교하고 결론의 한계를 말한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **센서 · 로그처럼 순서가 있는 데이터를 분류할 때** · 기준선 → RNN 계열 (1 · 2 · 6장)
- **`input.size(-1) must be equal to input_size` 에러를 만났을 때** · 세 겹 입력 · one-hot (3-1)
- **길이가 다른 문장 · 기록을 한 배치로 묶을 때** · 빈 칸 채우기 · `lens - 1` (3-4)
- **긴 줄에서 학습이 안 될 때** · 기울기 소실 · LSTM · GRU (4장)
- **모델 둘 중 하나를 골라 보고할 때** · 같은 조건 · 파라미터 · seed 비교 (6 · 7장)

## 학습 지도

<img src="images/rnn1_map.png" alt="RNN-1 학습 지도" width="733">

**예상 학습 시간** · 개념 2시간 + 실습 1시간 (GPU 전체 실행 약 4분 · CPU는 5 ~ 7장 HAR 학습에 수십 분)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">0</td><td>준비</td><td>라이브러리 · seed · 장치 · 수업 과제 데이터</td></tr>
<tr><td style="text-align: center;">1</td><td>순서가 중요한 데이터</td><td>합 · 평균 요약은 왜 순서 문제를 못 푸는가?</td></tr>
<tr><td style="text-align: center;">2</td><td>hidden state와 RNN</td><td>무엇을 넘기며, 왜 길이가 달라도 같은 모델인가?</td></tr>
<tr><td style="text-align: center;">3</td><td>입력 · 출력 모양</td><td><code>out</code> · <code>h_n</code>은 어떤 모양이고 어느 칸을 꺼내나?</td></tr>
<tr><td style="text-align: center;">4</td><td>멀리까지 닿는가</td><td>먼 신호는 왜 사라지고, LSTM · GRU는 무엇을 바꿨나?</td></tr>
<tr><td style="text-align: center;">5</td><td>실제 데이터</td><td>센서 기록을 RNN 입력으로 만들고 정규화하려면?</td></tr>
<tr><td style="text-align: center;">6</td><td>종합 실습</td><td>같은 조건에서 세 모델은 어떻게 다른가?</td></tr>
<tr><td style="text-align: center;">7</td><td>수업 과제 빈칸</td><td>크기 · seed · 순서 섞기를 바꾸면 결론이 흔들리나?</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

모든 장에서 쓰는 라이브러리 · seed · device · 수업 과제 데이터 · 공통 학습 함수를 한 번에 정의. **셀은 위에서부터 차례로 실행**(앞 섹션의 변수 · 함수를 이어서 씀).

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">이름</th><th style="text-align: left;">내용</th><th style="text-align: left;">처음 쓰는 곳</th></tr>
</thead>
<tbody>
<tr><td><code>set_seed()</code> · <code>DEV</code></td><td><code>random</code> · <code>torch</code> seed 고정, cuDNN 결정적 모드, GPU 없으면 CPU</td><td>전체</td></tr>
<tr><td><code>make_order_task()</code></td><td>숫자 10개 한 줄 · 3과 7이 하나씩 · 3이 먼저면 1 (수업 코드와 같은 데이터)</td><td>1-1</td></tr>
<tr><td><code>make_front_task()</code></td><td>3 · 7을 맨 앞 두 칸에 두고 뒤를 길게</td><td>4-1</td></tr>
<tr><td><code>to_tensor()</code></td><td>숫자 줄 → <code>(줄 수, 길이, 10)</code> one-hot 텐서</td><td>1-2</td></tr>
<tr><td><code>train()</code></td><td>미니배치 128 · Adam · 평가 정확도 반환</td><td>1-1</td></tr>
<tr><td><code>n_params()</code> · <code>kpad()</code></td><td>파라미터 개수 · 한글을 2칸 폭으로 세어 출력 줄 맞추기</td><td>1-2 · 3-4</td></tr>
<tr><td><code>xtr</code> · <code>xte</code></td><td>학습 6000줄 · 평가 2000줄</td><td>1-1</td></tr>
</tbody></table>

**실행 흐름**

1. 도구 불러오기
2. seed 고정 · 결정적 모드 · `set_seed()` 정의
3. 장치 정하기

In [1]:
# 라이브러리 불러오기
# 왜 · 이 노트북 전체에서 쓰는 도구를 한곳에 모아 두기 위해
# 왜 · 데이터 폴더 · 파일 경로를 다루고 실행 환경을 설정하기 위해
import os              # 폴더 · 파일 경로
import random          # 파이썬 난수 (데이터 만들기)
import statistics      # 평균 (시드별 정확도)
import time            # 학습 시간 재기
import urllib.request  # 데이터 내려받기 (5장)
import zipfile         # 압축 풀기 (5장)

import torch                     # 텐서 · 자동 미분
import torch.nn as nn            # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F  # F.one_hot (3-1)


# 재현성: seed 고정 + cuDNN 결정적 모드
# 왜 · 같은 환경에서 다시 실행해도 같은 숫자가 나와 설명과 출력을 맞추기 위해
SEED = 20260912                            # 수업 코드와 같은 seed
torch.backends.cudnn.deterministic = True  # 같은 GPU · 같은 버전에서 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False     # 실행마다 계산 방법을 새로 고르지 않음


# 왜 · 실험마다 같은 출발점에서 시작하도록 난수를 한 번에 다시 고정하기 위해
def set_seed(seed=SEED):     # seed를 한 번에 고정하는 함수
    random.seed(seed)        # 파이썬 random
    torch.manual_seed(seed)  # CPU · GPU 난수 모두


# GPU가 없으면 CPU로 실행
# 왜 · GPU 유무와 상관없이 같은 코드로 실행되게 하기 위해
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 없으면 CPU
print(f"torch {torch.__version__} · device {DEV.type}")             # 버전 · 장치 기록

torch 2.11.0+cu128 · device cuda


**실행 흐름**

1. 순서 과제 · 앞부분 과제 만드는 함수 정의
2. 숫자 줄 → 원-핫 텐서 함수 정의

In [2]:
# 수업 과제: 숫자 10개 한 줄에서 3이 7보다 먼저인가 (1 = 네, 0 = 아니요)
# 왜 · 순서가 답을 정하는 과제를 수업과 같은 규칙으로 만들기 위해
OTHERS = [d for d in range(10) if d not in (3, 7)]  # 3 · 7 을 뺀 숫자


# 왜 · 길이 · seed만 바꿔 같은 과제를 여러 번 만들기 위해
def make_order_task(n, length=10, seed=None):                # 3 · 7 순서 과제 줄 n개
    rng = random.Random(seed)                                # 이 데이터 전용 난수
    xs, ys = [], []                                          # 입력 줄 · 정답을 모을 목록
    for _ in range(n):                                       # 줄 n개 만들기
        s = [rng.choice(OTHERS) for _ in range(length - 2)]  # 채울 숫자
        i, j = sorted(rng.sample(range(length), 2))          # 3 · 7 이 들어갈 자리
        y = rng.randint(0, 1)                                # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                  # 정답 1이면 3이 앞
        s.insert(i, a)                                       # 앞 자리에 첫 숫자
        s.insert(j, b)                                       # 뒤 자리에 둘째 숫자
        xs.append(s)                                         # 완성한 줄 저장
        ys.append(y)                                         # 정답 저장
    return xs, ys                                            # 줄 목록 · 정답 목록


# 4장용: 3 · 7 을 맨 앞 두 칸에 두고 뒤는 상관없는 숫자
# 왜 · 멀리 기억해야 하는 과제를 따로 만들어 4장에서 쓰기 위해
def make_front_task(n, length, seed=None):   # 3 · 7을 맨 앞에 두는 과제 (4장)
    rng = random.Random(seed)                # 이 데이터 전용 난수 (전역 seed와 분리)
    xs, ys = [], []                          # 입력 줄 · 정답을 모을 목록
    for _ in range(n):                       # 줄 n개 만들기
        y = rng.randint(0, 1)                # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)  # 정답 1이면 3이 앞
        # 앞 두 칸 + 뒤는 상관없는 숫자
        xs.append([a, b] + [rng.choice(OTHERS) for _ in range(length - 2)])
        ys.append(y)                         # 정답 저장
    return xs, ys                            # 줄 목록 · 정답 목록


# 숫자 줄 → (줄 수, 길이, 10) 텐서 (숫자마다 10칸 중 한 칸만 1, 3-1)
# 왜 · 숫자 줄을 모델이 받는 원-핫 텐서로 바꾸기 위해
def to_tensor(xs):                                              # 숫자 줄 → one-hot 텐서
    return F.one_hot(torch.tensor(xs), num_classes=10).float()  # (줄, 길이) → (줄, 길이, 10)

**실행 흐름**

1. 공통 학습 함수 · 파라미터 수 · 한글 폭 맞춤 함수 정의
2. 학습 6000줄 · 평가 2000줄 만들기

In [3]:
# 공통 학습 함수: 미니배치 128 · Adam → 평가 정확도
# 왜 · 모든 비교가 같은 학습 절차 · 같은 평가 방법을 쓰게 하기 위해
def train(model, X, Y, Xe, Ye, epochs=30, lr=3e-3):               # 학습 → 평가 정확도
    model = model.to(DEV)                                         # 모델을 GPU로
    X, Y, Xe, Ye = X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)   # 데이터도 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)             # 가중치를 고칠 Adam
    loss_fn = nn.CrossEntropyLoss()                               # 분류 손실
    for _ in range(epochs):                                       # epoch 반복
        perm = torch.randperm(len(X)).to(DEV)                     # 매 epoch 순서 섞기
        for i in range(0, len(X), 128):                           # 128줄씩 미니배치
            j = perm[i:i + 128]                                   # 이번 배치 번호
            opt.zero_grad()                                       # 지난 기울기 지우기
            loss_fn(model(X[j]), Y[j]).backward()                 # 예측 → 손실 → 기울기
            opt.step()                                            # 가중치 한 걸음 수정
    model.eval()                                                  # 평가 모드
    with torch.no_grad():                                         # 평가는 기울기 기록 안 함
        return (model(Xe).argmax(1) == Ye).float().mean().item()  # 맞힌 비율 = 정확도


# 왜 · 모델 크기를 같은 방법으로 세어 비교하기 위해
def n_params(model):                                   # 파라미터 개수 세기
    return sum(p.numel() for p in model.parameters())  # 파라미터 개수


# 왜 · 한글 폭을 맞춰 여러 결과를 표처럼 읽기 위해
def kpad(text, width):  # 한글은 2칸 폭으로 세어 줄 맞추기
    return text + " " * (width - sum(2 if ord(ch) > 0x1100 else 1  # 모자란 폭만큼 공백
                                     for ch in text))


# 학습 6000줄 · 평가 2000줄 (수업 코드와 같은 데이터)
# 왜 · 수업과 같은 데이터로 결과를 그대로 비교하기 위해
xtr, ytr = make_order_task(6000, 10, seed=1)     # 학습 6000줄
xte, yte = make_order_task(2000, 10, seed=2)     # 평가 2000줄 (다른 seed)
Ytr, Yte = torch.tensor(ytr), torch.tensor(yte)  # 정답 → 텐서
print(f"학습 {len(xtr)}줄 · 평가 {len(xte)}줄 · 예시 {xtr[0]} → {ytr[0]}")  # 개수 · 예시 확인

학습 6000줄 · 평가 2000줄 · 예시 [2, 3, 1, 7, 5, 1, 9, 9, 9, 8] → 1


---
## 1. 순서가 중요한 데이터

> **이 장의 흐름**
>
> - 지금까지(MLP · CNN)는 한 샘플을 한 덩어리로 넣었다
> - 순서가 답을 정하는 데이터에서 덩어리 요약이 어떻게 실패하는지 먼저 확인

**파이프라인 위치** · ① 데이터 준비 · ⑥ 평가 — 순서가 중요한지 판단, 기준선 만들기

### 1-1. `A` 순서를 버리는 요약 · `sum()` · `mean` <sub>수업 코드 L0 · 강의 '순서가 중요하다'</sub>

> 🎯 **합 · 평균 같은 요약으로 '3이 7보다 먼저인가'를 풀면 왜 찍기 수준이 되는가?**

💡 **쉽게 말하면**

- 합 · 평균은 **장바구니 영수증의 총액**.
- 무엇을 먼저 담았는지는 영수증에 남지 않아서, 총액만 보고는 순서를 되돌릴 수 없음.

<img src="images/rnn1_order.png" alt="순서만 다른 두 줄" width="718">

**실행 흐름**

1. 순서만 다른 두 줄의 합 비교
2. 요약 4가지(합 · 평균 · 마지막 · 첫 값) + MLP 학습
3. 마지막 값이 60%인 이유 계산
4. 질문만 바꿔 보기

**먼저 예측** · 합 하나로 학습한 분류기의 정확도는 몇 %일까? (찍기는 50%) — 답은 코드 실행 결과 아래에

In [4]:
# 순서만 다른 두 줄: 합은 같고 답은 반대
# 왜 · 순서를 버리는 요약으로는 두 줄을 구분할 수 없음을 보이기 위해
a = [9, 3, 1, 0, 7, 5, 8, 6, 8, 4]                                           # 3이 먼저 → 1
b = [9, 7, 1, 0, 3, 5, 8, 6, 8, 4]                                           # 7이 먼저 → 0
print("합:", sum(a), sum(b), "| 정렬하면 같은 줄:", sorted(a) == sorted(b))  # 합 · 재료 비교


# 한 줄을 숫자 하나로 요약하는 함수 4가지
# 왜 · 사람이 정한 요약이 정답 정보를 얼마나 남기는지 비교하기 위해
def summarize(xs, how):                                       # 줄 → 숫자 하나
    if how == "sum":                                          # 합
        v = [sum(s) for s in xs]                              # 줄마다 합
    elif how == "mean":                                       # 평균
        v = [sum(s) / len(s) for s in xs]                     # 줄마다 평균
    elif how == "last":                                       # 마지막 값
        v = [s[-1] for s in xs]                               # 줄마다 마지막 칸
    else:                                                     # first
        v = [s[0] for s in xs]                                # 줄마다 첫 칸
    return torch.tensor(v, dtype=torch.float32).unsqueeze(1)  # (N,) → (N, 1)


# 요약값 → 분류기 (은닉층 depth개 · 너비 32 · 출력 2)
# 왜 · 요약 숫자만 받는 단순 모델로 요약의 한계를 확인하기 위해
class MLP(nn.Module):                                           # 요약값을 받는 분류기
    def __init__(self, n_in, width=32, depth=2):                # 입력 크기 · 너비 · 층 수
        super().__init__()                                      # nn.Module 준비 (필수)
        layers = [nn.Linear(n_in, width), nn.ReLU()]            # 첫 층
        for _ in range(depth - 1):                              # 은닉층 추가
            layers += [nn.Linear(width, width), nn.ReLU()]      # 같은 너비 층
        self.net = nn.Sequential(*layers, nn.Linear(width, 2))  # 마지막 출력 2 = 네 · 아니요

    def forward(self, x):   # 입력이 지나가는 길
        return self.net(x)  # 층을 차례로 통과


# 요약 4가지로 각각 학습 → 평가 정확도 (2-2에서 RNN과 비교)
# 왜 · 요약 방법마다 같은 조건으로 학습해 RNN과 비교할 기준선을 만들기 위해
base_acc = {}                                                # 기준선 정확도 모음 (2-2)
for how in ["sum", "mean", "last", "first"]:                 # 요약 4가지
    set_seed()                                               # 모델마다 같은 시작점
    base_acc[how] = train(MLP(1), summarize(xtr, how), Ytr,  # 요약값으로 학습
                          summarize(xte, how), Yte)
    print(f"{how:<5} → {base_acc[how]:.4f}")                 # 요약별 정확도


# 마지막 값이 60% 근처인 이유: 마지막 칸이 3 · 7 이면 답이 확정
# 왜 · 기준선 점수가 어디서 오는지 개수로 설명하기 위해
hit = sum(1 for s in xte if s[-1] in (3, 7)) / len(xte)  # 끝이 3 · 7인 줄 비율
print(f"마지막 칸이 3 또는 7: {hit:.3f} → 기대 정확도 "  # 확정 + 찍기 기대값
      f"{hit:.3f} × 1 + {1 - hit:.3f} × 0.5 = {hit + (1 - hit) / 2:.3f}")


# 같은 합 모델 · 같은 데이터로 질문만 바꾸기: 합이 45 이상인가
# 왜 · 같은 요약도 질문에 따라 충분할 수 있다는 것을 비교하기 위해
y45 = torch.tensor([int(sum(s) >= 45) for s in xtr])    # 새 질문의 정답 (학습)
y45e = torch.tensor([int(sum(s) >= 45) for s in xte])   # 새 질문의 정답 (평가)
set_seed()                                              # 모델마다 같은 시작점
acc45 = train(MLP(1), summarize(xtr, "sum"), y45,       # 같은 합 모델 · 다른 질문
              summarize(xte, "sum"), y45e)
print(f"질문을 '합 ≥ 45 인가'로 바꾸면 → {acc45:.4f}")  # 질문이 바뀐 정확도

합: 51 51 | 정렬하면 같은 줄: True


sum   → 0.4880


mean  → 0.4950


last  → 0.6065


first → 0.6085
마지막 칸이 3 또는 7: 0.211 → 기대 정확도 0.211 × 1 + 0.789 × 0.5 = 0.606


질문을 '합 ≥ 45 인가'로 바꾸면 → 0.9575


**결과** · 예측 정답은 `0.4880` — 찍기보다도 낮음. 합에는 '어느 쪽이 먼저인가'가 남아 있지 않음

- `합: 51 51 | 정렬하면 같은 줄: True` — 들어 있는 숫자는 똑같고 순서만 다름 → 답은 1과 0으로 반대
- `sum   → 0.4880` — 찍기(50%)보다도 낮음, 모델이 아니라 입력에 답이 없음
- `mean  → 0.4950` — 평균도 순서를 보지 않는 계산 → 찍기 수준
- `last  → 0.6065` — 마지막 칸이 3 · 7 인 줄만 확실히 맞힘
- `first → 0.6085` — 첫 칸도 같은 이유 (첫 칸이 3이면 답 1)
- `마지막 칸이 3 또는 7: 0.211 → 기대 정확도 0.211 × 1 + 0.789 × 0.5 = 0.606` — 21.1%는 확실히 맞히고 나머지는 찍기 → 실제 `0.6065`와 일치
- `질문을 '합 ≥ 45 인가'로 바꾸면 → 0.9575` — 같은 데이터 · 같은 합 모델인데 질문이 순서와 무관하면 잘 맞힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>합 · 평균 (요약 기준선)</td><td>순서를 버리고 한 줄을 <strong>숫자 하나</strong>로</td><td>순서와 상관없는 질문 (<code>합이 45 이상인가</code>)</td></tr>
<tr><td>마지막 값 · 첫 값</td><td><strong>한 칸</strong>만 남기고 나머지는 버림</td><td>답이 한쪽 끝에서 정해지는 문제</td></tr>
<tr><td>RNN (2-2)</td><td>한 칸씩 읽으며 넘길 요약을 <strong>학습</strong></td><td>순서가 답을 정하는 문제</td></tr>
<tr><td>순서 섞기 검사 (7-3)</td><td>한 줄 안의 칸 순서만 섞어 성능 비교</td><td>새 데이터에서 순서가 중요한지 <strong>먼저</strong> 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 모델을 키워도(층 · 너비 · epoch) 입력에 없는 정보는 생기지 않음
> - 성능이 안 나오면 먼저 입력에 답이 들어 있는지 확인

> ✅ **한 줄 정리** · 순서가 중요한지는 데이터가 아니라 질문이 정하며, 합 · 평균은 순서를 버리므로 순서가 답을 정하는 문제를 풀 수 없다.

**딥러닝 코드에서는**

- 새 시계열
- 텍스트 과제를 받으면 합
- 평균 기준선을 먼저 돌려 바닥값을 잡음 (HAR에서는 7-3 순서 섞기 검사)

### 1-2. `B` 넘길 것을 사람이 정하기 · `reshape(N, -1)` <sub>수업 코드 L0 '통째로 펼쳐서' · '봤는가' · 강의</sub>

> 🎯 **요약 하나가 부족하면 전부 넘기거나 필요한 것만 골라 넘기면 되지 않는가?**

💡 **쉽게 말하면**

- 전부 넘기기는 **이삿짐을 통째로 들고 다니기** — 길이가 길어질수록 짐(입력 · 파라미터)이 커짐.
- 필요한 것만 고르기는 정답을 아는 사람만 할 수 있음.

**실행 흐름**

1. 길이 10 · 50 · 100을 펼쳐서 MLP
2. 길이 50에서 데이터 500줄
3. '봤는가' 2칸
4. '먼저 봤는가' 3상태

**먼저 예측** · 길이 100을 통째로 펼치면 정확도는 길이 10(100%)보다 오를까, 떨어질까? — 답은 코드 실행 결과 아래에

In [5]:
# 방법 1: 한 줄 전체를 펼쳐서 한 번에 (길이 L → L × 10칸)
# 왜 · 모두 펼쳐 넣는 방식의 한계를 확인하기 위해
def flat(xs):                                  # 한 줄 전체를 펼침
    return to_tensor(xs).reshape(len(xs), -1)  # (N, L, 10) → (N, L*10)


# 왜 · 길이 · 데이터 양에 따라 펼친 MLP가 어떻게 무너지는지 비교하기 위해
for L, n_train in [(10, 6000), (50, 6000), (100, 6000), (50, 500)]:  # 길이 · 학습량 조합
    xa, ya = make_order_task(n_train, L, seed=1)                     # 학습 줄
    xb, yb = make_order_task(2000, L, seed=2)                        # 평가 줄
    set_seed()                                                       # 모델마다 같은 시작점
    model = MLP(L * 10)                                              # 입력 크기가 길이에 따라 바뀜
    acc = train(model, flat(xa), torch.tensor(ya), flat(xb), torch.tensor(yb))  # 펼친 입력으로 학습
    print(f"펼치기 · 길이 {L:>3} · 학습 {n_train:>4}줄 → {acc:.4f}"  # 정확도 · 파라미터
          f" · 파라미터 {n_params(model):,}")


# 방법 2: 끝까지 읽고 '3을 봤는가 · 7을 봤는가' 두 칸만 넘기기
# 왜 · 봤는지만으로는 순서를 알 수 없다는 것을 확인하기 위해
def seen_two(xs):                                                      # 봤는가 2칸
    return torch.tensor([[float(3 in s), float(7 in s)] for s in xs])  # 3 있음 · 7 있음


set_seed()                                                            # 모델마다 같은 시작점
acc = train(MLP(2, depth=1), seen_two(xtr), Ytr, seen_two(xte), Yte,  # 2칸으로 학습
            epochs=40)
print(f"봤는가 2칸 → {acc:.4f}")                                      # 정확도


# 방법 3: 상태 3개 (둘 다 안 봄 · 3이 먼저 · 7이 먼저) 중 끝 상태 넘기기
# 왜 · 무엇을 넘기느냐가 정답을 가른다는 것을 상태 셋으로 보이기 위해
def first_seen(xs):                                                  # 끝 상태 3가지
    return torch.tensor([[0.0, 1.0, 0.0] if s.index(3) < s.index(7)  # 3이 먼저면 [0, 1, 0]
                         else [0.0, 0.0, 1.0] for s in xs])


set_seed()                                                                # 모델마다 같은 시작점
acc = train(MLP(3, depth=1), first_seen(xtr), Ytr, first_seen(xte), Yte,  # 3상태로 학습
            epochs=40)
print(f"먼저 봤는가 3상태 → {acc:.4f}")                                   # 정확도

펼치기 · 길이  10 · 학습 6000줄 → 1.0000 · 파라미터 4,354


펼치기 · 길이  50 · 학습 6000줄 → 0.9585 · 파라미터 17,154


펼치기 · 길이 100 · 학습 6000줄 → 0.8840 · 파라미터 33,154
펼치기 · 길이  50 · 학습  500줄 → 0.6695 · 파라미터 17,154


봤는가 2칸 → 0.5120


먼저 봤는가 3상태 → 1.0000


**결과** · 예측 정답은 떨어짐 — `0.8840`. 입력이 1000칸 · 파라미터 33,154개로 커져 같은 데이터로는 배우기 어려움

- `펼치기 · 길이  10 · 학습 6000줄 → 1.0000 · 파라미터 4,354` — 짧을 때는 펼쳐도 100%
- `펼치기 · 길이 100 · 학습 6000줄 → 0.8840 · 파라미터 33,154` — 길어질수록 입력 · 파라미터가 늘고 정확도는 떨어짐 (길이 50은 `0.9585`)
- `펼치기 · 길이  50 · 학습  500줄 → 0.6695` — 같은 길이 50도 데이터가 적으면 67% — 배울 것(17,154개)에 비해 볼 기회가 부족
- `봤는가 2칸 → 0.5120` — 3도 7도 늘 둘 다 들어 있음 → 두 칸이 항상 같은 값 → 찍기
- `먼저 봤는가 3상태 → 1.0000` — 필요한 정보를 정확히 넘기면 100% (단 사람이 규칙을 알 때만)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>통째로 펼치기 (MLP)</td><td>칸마다 <strong>다른</strong> 가중치, 길이가 바뀌면 모델도 바뀜</td><td>길이가 짧고 고정, 데이터가 충분할 때</td></tr>
<tr><td>사람이 정한 상태 (3상태)</td><td>필요한 정보를 사람이 <strong>설계</strong></td><td>규칙을 정확히 알 때만 (실제 데이터는 거의 불가)</td></tr>
<tr><td>RNN (2장)</td><td>모든 칸이 <strong>같은</strong> 가중치, 넘길 것은 학습</td><td>길이가 길거나 매번 다를 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 펼치기는 학습할 때의 길이로 모델 크기가 고정됨
> - 더 긴 줄이 오면 넣을 수 없음

> ✅ **한 줄 정리** · 넘길 것을 사람이 정하면 맞을 때도 틀릴 때도 있으므로, 크기만 고정하고 내용은 학습에 맡기는 것이 RNN이다.

**딥러닝 코드에서는**

- 이미지에서 특징을 사람이 정하지 않고 CNN이 배우듯, 순서 데이터에서는 넘길 요약을 RNN이 배움

---
## 2. hidden state와 RNN

> **이 장의 흐름**
>
> - 1장에서 넘길 것을 사람이 정하면 실패하거나 길이에 묶였다
> - 크기만 고정하고 내용은 학습하는 요약(hidden state)을 칸마다 같은 상자로 넘김

**파이프라인 위치** · ④ 모델 · ⑥ 평가 — `nn.RNN` 정의, 기준선과 비교

### 2-1. `A` hidden state 손으로 펼치기 · `nn.RNNCell` <sub>수업 코드 L1 '손으로 펼친 판' · 'nn.RNN 과 맞춰 보기'</sub>

> 🎯 **hidden state는 무엇이고, `nn.RNN` 안에서는 어떤 계산이 반복되는가?**

💡 **쉽게 말하면**

- hidden state는 **매 칸 새로 쓰는 32칸짜리 메모**.
- 이번 숫자와 직전 메모를 같은 상자(같은 가중치)에 넣어 새 메모를 쓰고, 길이가 몇이든 상자는 하나.

<img src="images/rnn1_hidden_flow.png" alt="hidden state 시간축 흐름" width="627">

**실행 흐름**

1. `RNNCell` 하나 + `for` 10번으로 요약 만들기
2. `nn.RNN` 결과와 비교
3. 길이 10 · 50 · 200에서 파라미터 수
4. 손계산

**먼저 예측** · `nn.RNN`과 손으로 편 `for` 문의 결과는 얼마나 다를까? — 답은 코드 실행 결과 아래에

In [6]:
# 상자 하나(RNNCell) + for 문 = 손으로 펼친 RNN
# 왜 · RNN이 하는 일을 for 문으로 직접 펼쳐 이해하기 위해
set_seed()                                  # 모델마다 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)      # PyTorch가 감싸 둔 판
cell = nn.RNNCell(10, 32)                   # 상자 하나: 입력 10 · 요약 32
with torch.no_grad():                       # 비교하려고 같은 가중치를 복사
    cell.weight_ih.copy_(rnn.weight_ih_l0)  # 입력 가중치 복사
    cell.weight_hh.copy_(rnn.weight_hh_l0)  # 이전 요약 가중치 복사
    cell.bias_ih.copy_(rnn.bias_ih_l0)      # bias 복사
    cell.bias_hh.copy_(rnn.bias_hh_l0)      # bias 복사

# 왜 · 작은 배치로 칸마다 요약이 어떻게 바뀌는지 따라가기 위해
X = to_tensor(xtr[:4])                        # 4줄 · 10칸 · 칸마다 10개 → (4, 10, 10)
h = torch.zeros(4, 32)                        # 처음 요약 = 0
hs = []                                       # 칸마다 요약을 모음
for t in range(10):                           # 10칸을 차례로
    h = cell(X[:, t, :], h)                   # 이번 칸 + 직전 요약 → 새 요약 (덮어쓰기)
    hs.append(h)                              # 이번 칸 요약 저장
print("손으로 편 마지막 h:", tuple(h.shape))  # 마지막 요약 모양


# nn.RNN 결과와 비교
# 왜 · 직접 펼친 결과와 nn.RNN 결과가 같은지 검증하기 위해
output, h_n = rnn(X)             # 같은 입력을 nn.RNN에
manual = torch.stack(hs, dim=1)  # 요약 10개 → (4, 10, 32)
print("output 과 손 계산의 최대 차이:", (output - manual).abs().max().item())  # 0이면 같은 계산


# 분류 모델: RNN → 마지막 칸 요약 → Linear(32 → 2)
# 왜 · 넘길 요약을 학습으로 정하는 분류 모델을 만들기 위해
class Net(nn.Module):                              # RNN 분류 모델
    def __init__(self, kind="RNN", hid=32):        # 셀 종류 · 요약 크기
        super().__init__()                         # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(10, hid, batch_first=True)  # 입력 10 · (배치, 길이, 입력)
        self.o = nn.Linear(hid, 2)                 # 요약 → 점수 2개

    def forward(self, x):             # 입력이 지나가는 길
        out, _ = self.r(x)            # out: (N, L, hid)
        return self.o(out[:, -1, :])  # 마지막 칸 → (N, 2)


# 파라미터 공유: 길이가 달라도 같은 모델 · 같은 파라미터 수
# 왜 · 길이와 상관없이 같은 가중치를 칸마다 쓴다는 것을 확인하기 위해
net = Net("RNN")                                                   # 학습 전 모델 하나
for L in [10, 50, 200]:                                            # 길이만 바꿔
    xs, _ = make_order_task(4, L, seed=3)                          # 길이 L인 4줄
    print(f"길이 {L:>3} → 출력 {tuple(net(to_tensor(xs)).shape)}"  # 출력 · 파라미터 수
          f" · 파라미터 {n_params(net):,}")
print("손계산:", 32 * (10 + 32) + 32 + 32 + 32 * 2 + 2)            # RNN + Linear

손으로 편 마지막 h: (4, 32)
output 과 손 계산의 최대 차이: 0.0
길이  10 → 출력 (4, 2) · 파라미터 1,474
길이  50 → 출력 (4, 2) · 파라미터 1,474
길이 200 → 출력 (4, 2) · 파라미터 1,474
손계산: 1474


**결과** · 예측 정답은 `0.0` — 완전히 같음. `nn.RNN`은 같은 `for` 문을 안에서 돌릴 뿐

- `손으로 편 마지막 h: (4, 32)` — 4줄 × 요약 32칸, 길이 10 축은 사라지고 마지막 요약만 남음
- `output 과 손 계산의 최대 차이: 0.0` — `nn.RNN` 안에서도 같은 계산 `h = tanh(W·x + U·h + b)`가 반복됨
- `길이 200 → 출력 (4, 2) · 파라미터 1,474` — 길이 10 · 50 · 200 모두 같은 모델 · 같은 1,474개 (파라미터 공유)
- `손계산: 1474` — RNN 32 × 42 + 32 + 32 = 1408, Linear 32 × 2 + 2 = 66, 합 1474

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>nn.RNNCell</code> + <code>for</code></td><td>한 칸씩 직접 돌림, 중간 <code>h</code>를 잡을 수 있음</td><td>기울기 측정 (4-1) · 칸마다 다른 처리</td></tr>
<tr><td><code>nn.RNN</code></td><td><code>for</code> 문을 안에서 처리 (cuDNN으로 한 번에 계산)</td><td>보통 학습 (<strong>기본 선택</strong>)</td></tr>
<tr><td>hidden state <code>h</code></td><td>매 칸 <strong>새로 쓰는</strong> 고정 크기 요약 (쌓이는 창고 X)</td><td>다음 칸으로 넘기는 것</td></tr>
<tr><td>파라미터 공유</td><td>모든 칸이 같은 <code>W</code>를 씀 (CNN 필터와 같은 발상)</td><td>길이와 상관없이 같은 모델 · 같은 파라미터 수</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - hidden state는 지금까지 본 것이 쌓이는 창고가 아님
> - 매 칸 덮어써지는 요약이라 오래된 정보는 흐려질 수 있음 (4장)

> ✅ **한 줄 정리** · RNN은 같은 가중치의 상자 하나로 `새 요약 = f(이전 요약, 이번 입력)`을 칸마다 반복하므로 길이와 상관없이 파라미터 수가 같다.

**딥러닝 코드에서는**

- `h_t = tanh(W_ih x_t + b_ih + W_hh h_(t-1) + b_hh)`
- PyTorch 문서의 `nn.RNN` 식, `weight_ih_l0` · `weight_hh_l0`가 각각 `W_ih` · `W_hh`

### 2-2. `A` RNN 학습과 기준선 비교 · `nn.RNN` <sub>수업 코드 L1 '학습시켜 보기' · 강의</sub>

> 🎯 **같은 데이터 · 같은 학습 함수에서 RNN은 합 · 평균 기준선보다 얼마나 나은가?**

💡 **쉽게 말하면**

- 기준선은 **시험 전에 찍어 본 점수**.
- 새 모델이 기준선을 확실히 넘어야 순서를 배웠다고 말할 수 있음.

**실행 흐름**

1. 1-1과 같은 데이터 · `train()` · seed로 RNN 학습
2. 기준선 4개와 나란히 출력

**먼저 예측** · RNN은 몇 %일까? — 답은 코드 실행 결과 아래에

In [7]:
# RNN 학습 (1-1과 같은 데이터 · 같은 학습 함수 · 같은 seed)
# 왜 · 기준선과 같은 조건에서 RNN만 바꿔 공정하게 비교하기 위해
set_seed()                                                             # 모델마다 같은 시작점
rnn_acc = train(Net("RNN"), to_tensor(xtr), Ytr, to_tensor(xte), Yte)  # 1-1과 같은 조건


# 기준선 4개와 나란히 (# 1개 = 5%)
# 왜 · RNN 점수를 기준선과 한눈에 비교하기 위해
for name, acc in [*base_acc.items(), ("RNN", rnn_acc)]:    # 기준선 4개 + RNN
    print(f"{name:<5} {acc:.4f} {'#' * round(acc * 20)}")  # 정확도 막대

sum   0.4880 ##########
mean  0.4950 ##########
last  0.6065 ############
first 0.6085 ############
RNN   1.0000 ####################


**결과** · 예측 정답은 `1.0000` — 사람이 넘길 것을 정해 준 1-2의 3상태와 같은 100%를, 넘길 것을 스스로 배워서 냄

- `sum   0.4880 ##########` — 합 기준선 — 찍기 수준
- `last  0.6065 ############` — 사람이 정한 요약 중 가장 높아도 61% 근처
- `RNN   1.0000 ####################` — 같은 데이터 · 같은 학습 함수인데 RNN만 전부 맞힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>합 · 평균 기준선</td><td><code>0.4880</code> · <code>0.4950</code> → 찍기 수준</td><td>어떤 모델이든 비교할 <strong>바닥값</strong>으로 먼저 돌려 봄</td></tr>
<tr><td>마지막 값 · 첫 값 기준선</td><td><code>0.6065</code> · <code>0.6085</code></td><td>끝 한 칸만으로 얼마나 되는지 확인</td></tr>
<tr><td>RNN</td><td><code>1.0000</code> · 파라미터 1,474개</td><td>기준선을 확실히 넘어야 순서를 배운 것</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 기준선 없이 RNN 점수만 보면 잘한 것인지 판단 불가
> - 비교표에 항상 기준선을 같이 적기

> ✅ **한 줄 정리** · 같은 조건에서 RNN이 합 · 평균 · 끝값 기준선을 크게 넘으면, 그 차이가 순서에서 배운 몫이다.

**딥러닝 코드에서는**

- 시계열 예측(RNN-2)에서는 '직전 값 그대로'
- '이동 평균' 기준선, 텍스트 분류(RNN-3)에서는 '가장 많은 클래스' 기준선과 비교

---
## 3. 입력 · 출력 모양

> **이 장의 흐름**
>
> - 2장에서 RNN을 만들었다
> - 실제로 짜면 가장 먼저 막히는 곳이 모양.
> - 넣는 세 겹, 축 순서, 돌려받는 `out` · `h_n`, 길이가 다른 줄을 정리

**파이프라인 위치** · ② 텐서로 · ④ 모델 — 입력 모양 맞추기, 어느 칸을 분류기에 넘길지

### 3-1. `A` 세 겹 입력과 one-hot · `F.one_hot` <sub>수업 코드 '모양 맞추기 — one_hot · batch_first' · 강의</sub>

> 🎯 **RNN 입력은 왜 세 겹 `(배치, 길이, 입력 크기)`이며, 숫자 줄은 어떻게 세 겹으로 만드는가?**

💡 **쉽게 말하면**

- 세 겹 = **몇 줄 · 한 줄에 몇 칸 · 칸 하나를 숫자 몇 개로 적나**.
- 숫자 3을 '3' 한 칸으로 적지 않고 10칸 중 3번 칸만 켜서 적는 것이 one-hot.

**실행 흐름**

1. 숫자 그대로
2. `unsqueeze`로 축만 붙여 에러 확인
3. `one_hot`으로 10칸 · 통과

**먼저 예측** · `(1, 10, 1)`을 `nn.RNN(10, 32)`에 넣으면? — 답은 코드 실행 결과 아래에

In [8]:
# 왜 · 모양 에러를 재현할 가장 작은 입력을 만들기 위해
digits = torch.tensor([xtr[0]])         # 숫자 그대로 → (1, 10)
rnn = nn.RNN(10, 32, batch_first=True)  # 칸마다 숫자 10개를 기대


# 축만 붙이면: 칸마다 숫자 1개 → 에러
# 왜 · 칸마다 숫자 하나만 넣을 때의 모양 에러를 미리 보기 위해
x_bad = digits.unsqueeze(-1).float()                              # (1, 10) → (1, 10, 1)
try:
    rnn(x_bad)                                                    # 칸마다 1개짜리 입력
except RuntimeError as e:                                         # 에러를 잡아 멈추지 않게
    print(tuple(x_bad.shape), "→ 에러:", str(e).splitlines()[0])  # 에러 첫 줄 읽기


# one_hot: 숫자 하나를 10칸에 적기 (그 자리만 1)
# 왜 · 범주 숫자를 원-핫으로 바꿔 모양을 맞추기 위해
x = F.one_hot(digits, num_classes=10).float()           # (1, 10) → (1, 10, 10)
output, h_n = rnn(x)                                    # one-hot 입력은 통과
print(tuple(x.shape), "→ output", tuple(output.shape),  # 입력 · 출력 모양
      "· h_n", tuple(h_n.shape))
print("숫자", xtr[0][1], "→", x[0, 1].int().tolist())   # 숫자 3의 one-hot

(1, 10, 1) → 에러: input.size(-1) must be equal to input_size. Expected 10, got 1
(1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)
숫자 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


**결과** · 예측 정답은 `RuntimeError` — 마지막 겹(칸마다 숫자 수)이 `input_size=10`과 달라서

- `(1, 10, 1) → 에러: input.size(-1) must be equal to input_size. Expected 10, got 1` — 세 겹은 맞지만 마지막 겹이 1 ≠ 10
- `(1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)` — one_hot으로 칸마다 10개 → 통과
- `숫자 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]` — 3번 자리만 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>unsqueeze(-1)</code></td><td>축만 추가, 칸마다 값 <strong>1개</strong> 그대로</td><td>값 자체가 크기를 가진 연속 수치 (센서 · 가격, <code>input_size=1</code>)</td></tr>
<tr><td><code>F.one_hot</code></td><td>번호 → 그 자리만 1인 벡터 (크기 = 종류 수)</td><td>범주 번호 (숫자 · 글자), 종류가 적을 때</td></tr>
<tr><td><code>nn.Embedding</code> (RNN-3)</td><td>번호 → <strong>학습되는</strong> 짧은 벡터</td><td>종류가 많을 때 (단어 수천 개)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 에러가 나면 먼저 `x.shape`를 출력해 세 번째 겹이 `input_size`와 같은지 확인

> ✅ **한 줄 정리** · RNN 입력은 `(배치, 길이, 입력 크기)` 세 겹이고, 범주 번호는 `one_hot`으로 마지막 겹을 `input_size`에 맞춘다.

**딥러닝 코드에서는**

- HAR 센서는 한 칸에 숫자 9개 → `(N, 128, 9)`
- `input_size=9`, 단어는 RNN-3에서 `nn.Embedding`으로 같은 일을 함

### 3-2. `A` 축 순서 · `batch_first` <sub>수업 코드 '모양 맞추기' · 레이어 실습 RNN</sub>

> 🎯 **`batch_first`는 무엇을 바꾸며, 잘못 두면 왜 에러 없이 틀리는가?**

💡 **쉽게 말하면**

- `batch_first`는 **앞 두 겹의 읽는 순서 약속**.
- 약속을 어기면 8줄 · 10칸을 10줄 · 8칸으로 읽어도 모양 계산이 맞아 떨어져 에러가 나지 않음.

**실행 흐름**

1. 같은 `(8, 10, 10)`을 True · False로
2. 레이어 실습의 False 입력 `(길이, 배치, 입력)`
3. `transpose(0, 1)`로 바꾸기

**먼저 예측** · `batch_first=False`로 `(8, 10, 10)`을 넣으면 `h_n`의 가운데 숫자는? — 답은 코드 실행 결과 아래에

In [9]:
# 같은 (8, 10, 10) 텐서를 True · False 로 넣기
# 왜 · 축 순서를 잘못 두면 에러 없이 틀린 결과가 나오는 함정을 비교하기 위해
x8 = torch.randn(8, 10, 10)                                        # 8줄 · 10칸 · 10개
for bf in [True, False]:                                           # 두 설정 비교
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x8)               # 같은 텐서를 넣음
    print(f"batch_first={bf!s:<5} → output {tuple(output.shape)}"  # 모양 비교
          f" · h_n {tuple(h_n.shape)}")


# 레이어 실습: 기본값 False 는 (길이, 배치, 입력) 순서
# 왜 · 기본값이 어떤 축 순서를 기대하는지 확인하기 위해
rnn2 = nn.RNN(input_size=10, hidden_size=4, num_layers=2)  # 레이어 실습 설정
x = torch.randn(3, 6, 10)                                  # 길이 3 · 배치 6 · 입력 10
out, h_n = rnn2(x)                                         # (길이, 배치, 입력)으로 넣음
print("False 마지막 칸 out[-1]:", tuple(out[-1].shape),    # 첫 축이 길이
      "== h_n[-1]:", torch.allclose(out[-1], h_n[-1]))


# 배치를 앞으로 옮기면 batch_first=True 와 같은 모양
# 왜 · transpose로 맞추는 방법도 같은 결과인지 확인하기 위해
x_bf = x.transpose(0, 1)                                           # (3, 6, 10) → (6, 3, 10)
print("transpose(0, 1):", tuple(x.shape), "→", tuple(x_bf.shape))  # 축 바꾼 결과

batch_first=True  → output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False → output (8, 10, 32) · h_n (1, 10, 32)
False 마지막 칸 out[-1]: (6, 4) == h_n[-1]: True
transpose(0, 1): (3, 6, 10) → (6, 3, 10)


**결과** · 예측 정답은 `10` — 8을 길이로, 10을 줄 수로 바꿔 읽음

- `batch_first=True  → output (8, 10, 32) · h_n (1, 8, 32)` — 8을 줄 수로 읽음 (의도대로)
- `batch_first=False → output (8, 10, 32) · h_n (1, 10, 32)` — 에러 없이 8을 길이 · 10을 줄 수로 바꿔 읽음
- `False 마지막 칸 out[-1]: (6, 4) == h_n[-1]: True` — False이면 첫 축이 길이 → 마지막 칸은 `out[-1]`
- `transpose(0, 1): (3, 6, 10) → (6, 3, 10)` — 앞 두 축만 바꾸면 `batch_first=True`용 모양

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>batch_first=True</code></td><td>입력 · out이 <strong>(배치, 길이, 입력)</strong></td><td><code>DataLoader</code> 배치와 같은 순서 → <strong>기본 선택</strong></td></tr>
<tr><td><code>batch_first=False</code> (기본값)</td><td>입력 · out이 <strong>(길이, 배치, 입력)</strong></td><td>옛 코드 · 공식 문서 예제를 읽을 때</td></tr>
<tr><td><code>h_n</code> 모양</td><td>설정과 상관없이 항상 <strong>(층 × 방향, 배치, H)</strong></td><td><code>h_n</code>은 배치가 항상 두 번째 → 마지막 층은 <code>h_n[-1]</code></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `batch_first`의 기본값은 `False`
> - 모델을 만들 때마다 `batch_first=True`를 직접 적기

> ✅ **한 줄 정리** · `batch_first=True`면 `(배치, 길이, 입력)`, `False`면 `(길이, 배치, 입력)`이며 `h_n`은 둘 다 배치가 두 번째다.

**딥러닝 코드에서는**

- `DataLoader`가 주는 배치는 `(배치, ...)` 순서 → `nn.LSTM(..., batch_first=True)`가 거의 모든 프로젝트 코드의 기본

### 3-3. `A` out · h_n 모양 손계산 · `num_layers` · `bidirectional` <sub>레이어 실습 RNN · LSTM · GRU · 강의 'Bidirectional'</sub>

> 🎯 **층 수 · 방향이 바뀌면 `out`과 `h_n`의 모양은 어떻게 바뀌는가?**

💡 **쉽게 말하면**

- `out`은 **마지막 층이 칸마다 쓴 요약 전부**, `h_n`은 **층마다 마지막 칸 요약 하나씩**.
- 양방향이면 칸마다 앞 · 뒤 요약을 옆으로 붙여 2배.

<img src="images/rnn1_shapes.png" alt="out은 맨 위 줄, h_n은 오른쪽 끝 열" width="718">

**손계산표** · 배치 6 · 길이 3 · 입력 10 · hidden 4, `batch_first=True` (코드 실행 전에 빈칸 없이 채워 보기)

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">설정</th><th style="text-align: left;">입력 x</th><th style="text-align: left;">out</th><th style="text-align: left;">h_n</th></tr>
</thead>
<tbody>
<tr><td>RNN 층 1</td><td>(6, 3, 10)</td><td>(6, 3, 4)</td><td>(1, 6, 4)</td></tr>
<tr><td>RNN 층 2</td><td>(6, 3, 10)</td><td>(6, 3, 4)</td><td>(2, 6, 4)</td></tr>
<tr><td>RNN 층 1 · 양방향</td><td>(6, 3, 10)</td><td>(6, 3, 8)</td><td>(2, 6, 4)</td></tr>
<tr><td>RNN 층 2 · 양방향</td><td>(6, 3, 10)</td><td>(6, 3, 8)</td><td>(4, 6, 4)</td></tr>
<tr><td>LSTM 층 2</td><td>(6, 3, 10)</td><td>(6, 3, 4)</td><td>(2, 6, 4) · c_n (2, 6, 4)</td></tr>
<tr><td>GRU 층 2</td><td>(6, 3, 10)</td><td>(6, 3, 4)</td><td>(2, 6, 4)</td></tr>
<tr><td>규칙</td><td>(배치, 길이, 입력)</td><td>(배치, 길이, <strong>방향 × H</strong>)</td><td>(<strong>층 × 방향</strong>, 배치, H)</td></tr>
</tbody></table>

**실행 흐름**

1. 손계산 규칙 함수
2. 6가지 설정을 실제로 만들어 손계산과 비교
3. 마지막 칸 `out`과 `h_n` 값 비교 (한 방향 · 양방향)

**먼저 예측** · `nn.RNN(10, 4, num_layers=2, bidirectional=True)`에 `(6, 3, 10)`을 넣으면 `h_n`은? (아래 손계산 표를 먼저 채우기) — 답은 코드 실행 결과 아래에

In [10]:
# 손계산 규칙: out = (배치, 길이, 방향 × H) · h_n = (층 × 방향, 배치, H)
# 왜 · 실행 전에 출력 모양을 규칙으로 계산할 수 있게 하기 위해
def expect(N, L, H, layers=1, bi=False):      # 손계산 규칙
    d = 2 if bi else 1                        # 방향 수
    return (N, L, d * H), (layers * d, N, H)  # out 모양 · h_n 모양


# 레이어 실습 크기: 배치 6 · 길이 3 · 입력 10 · hidden 4
# 왜 · 손계산한 모양과 실제 모양을 여러 조합으로 대조하기 위해
N, L, I, H = 6, 3, 10, 4                                          # 배치 · 길이 · 입력 · hidden
x = torch.randn(N, L, I)                                          # (6, 3, 10) 입력
cases = [("RNN", 1, False), ("RNN", 2, False), ("RNN", 1, True),  # 층 · 방향 조합 6가지
         ("RNN", 2, True), ("LSTM", 2, False), ("GRU", 2, False)]
for kind, layers, bi in cases:                                    # 조합마다
    layer = getattr(nn, kind)(I, H, num_layers=layers, bidirectional=bi,  # 층 만들기
                              batch_first=True)
    out, h_n = layer(x)                                           # 실제 모양
    if kind == "LSTM":                                            # LSTM만 두 번째가 튜플
        h_n, c_n = h_n                                            # LSTM 은 (h_n, c_n) 묶음
    ok = (tuple(out.shape), tuple(h_n.shape)) == expect(N, L, H, layers, bi)  # 손계산과 비교
    print(f"{kind:<4} 층 {layers} 양방향 {bi!s:<5} → out {tuple(out.shape)}"  # 결과 한 줄
          f" · h_n {tuple(h_n.shape)} · 손계산 일치 {ok}")


# 마지막 칸 out == 마지막 층 h_n ? (한 방향 · 양방향)
# 왜 · 두 출력이 언제 같고 언제 다른지 확인하기 위해
uni = nn.RNN(I, H, num_layers=2, batch_first=True)                             # 한 방향 2층
out, h_n = uni(x)                                                              # 한 방향 결과
print("한 방향: out[:, -1] == h_n[-1]:", torch.allclose(out[:, -1], h_n[-1]))  # 같은 값인지
bi = nn.RNN(I, H, bidirectional=True, batch_first=True)                        # 양방향 1층
out, h_n = bi(x)                                                               # 양방향 결과
print("양방향 : out[:, -1, H:] == h_n[-1]:",                                   # 뒤 방향 끝은 0번 칸
      torch.allclose(out[:, -1, H:], h_n[-1]),
      "| out[:, 0, H:] == h_n[-1]:", torch.allclose(out[:, 0, H:], h_n[-1]))

RNN  층 1 양방향 False → out (6, 3, 4) · h_n (1, 6, 4) · 손계산 일치 True
RNN  층 2 양방향 False → out (6, 3, 4) · h_n (2, 6, 4) · 손계산 일치 True
RNN  층 1 양방향 True  → out (6, 3, 8) · h_n (2, 6, 4) · 손계산 일치 True
RNN  층 2 양방향 True  → out (6, 3, 8) · h_n (4, 6, 4) · 손계산 일치 True
LSTM 층 2 양방향 False → out (6, 3, 4) · h_n (2, 6, 4) · 손계산 일치 True
GRU  층 2 양방향 False → out (6, 3, 4) · h_n (2, 6, 4) · 손계산 일치 True
한 방향: out[:, -1] == h_n[-1]: True
양방향 : out[:, -1, H:] == h_n[-1]: False | out[:, 0, H:] == h_n[-1]: True


**결과** · 예측 정답은 `(4, 6, 4)` — 층 2 × 방향 2 = 4

- `RNN  층 2 양방향 False → out (6, 3, 4) · h_n (2, 6, 4) · 손계산 일치 True` — 층이 늘면 h_n 첫 축만 늘어남 (out은 마지막 층만)
- `RNN  층 1 양방향 True  → out (6, 3, 8) · h_n (2, 6, 4) · 손계산 일치 True` — 양방향이면 out 마지막 축이 2 × 4 = 8, h_n 첫 축이 2
- `RNN  층 2 양방향 True  → out (6, 3, 8) · h_n (4, 6, 4) · 손계산 일치 True` — 층 2 × 방향 2 → h_n 첫 축 4
- `LSTM 층 2 양방향 False → out (6, 3, 4) · h_n (2, 6, 4) · 손계산 일치 True` — LSTM · GRU도 같은 규칙 (LSTM은 같은 모양의 `c_n`이 하나 더)
- `한 방향: out[:, -1] == h_n[-1]: True` — 한 방향이면 마지막 칸 출력 = 마지막 층 h_n
- `양방향 : out[:, -1, H:] == h_n[-1]: False | out[:, 0, H:] == h_n[-1]: True` — 뒤 방향은 끝에서 출발해 0번 칸에서 끝남 → 0번 칸의 뒤쪽 절반이 `h_n[-1]`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>num_layers=2</code></td><td>RNN을 위로 쌓음, out은 <strong>마지막 층만</strong>, h_n은 층마다</td><td>패턴이 복잡할 때 (보통 1 ~ 3층)</td></tr>
<tr><td><code>bidirectional=True</code></td><td>앞 → 뒤 + 뒤 → 앞, out 마지막 축 <strong>2배</strong></td><td>줄 전체가 이미 있을 때 (문장 분류 · HAR 창)</td></tr>
<tr><td>한 방향 (기본)</td><td>지금까지 본 것만 사용</td><td>실시간 판정 · 다음 값 예측 (뒤가 아직 없음)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 양방향에서 `out[:, -1]`은 앞 방향만 끝까지 읽은 값
> - 분류에는 `h_n[-2]`(앞)
> - `h_n[-1]`(뒤)를 이어 붙여 씀

> ✅ **한 줄 정리** · `out`은 `(배치, 길이, 방향 × H)`, `h_n`은 `(층 × 방향, 배치, H)`이고, 한 방향일 때만 `out[:, -1]`과 `h_n[-1]`이 같다.

**딥러닝 코드에서는**

- `out, (h, c) = self.lstm(x)` → `self.fc(h[-1])` 또는 `self.fc(out[:, -1])` · 한 방향
- 길이가 같으면 둘이 같은 값

### 3-4. `A` 마지막 시점 vs 전체 시점 · `out[idx, lens - 1]` <sub>수업 코드 L1b '빈 칸을 읽으면' · 강의 padding</sub>

> 🎯 **길이가 다른 줄을 한 텐서에 담으면, 요약은 어느 칸에서 꺼내야 하는가?**

💡 **쉽게 말하면**

- 짧은 줄은 **빈 칸(0)으로 채운 틀**에 들어감.
- `-1`은 틀의 끝이지 데이터의 끝이 아니라서, 빈 칸을 몇 번 더 지난 요약을 읽게 됨.

<img src="images/rnn1_padding.png" alt="빈 칸 채우기와 진짜 끝" width="686">

**실행 흐름**

1. 3줄을 6칸 틀에 채우기
2. `-1`과 `lens - 1` 비교
3. 전체 시점 출력 모양
4. 길이 5~60 줄로 꺼내는 방법 4가지 학습

**먼저 예측** · 길이 5~60인 줄에서 `out[:, -1]`을 읽으면 몇 %일까? (길이 10 고정일 때는 100%) — 답은 코드 실행 결과 아래에

In [11]:
# 길이 3 · 6 · 4 인 세 줄 → 가장 긴 6칸 틀에 빈 벡터(전부 0)로 채우기
# 왜 · 길이가 다른 줄을 한 텐서로 묶되 진짜 길이를 함께 들고 다니기 위해
seqs = [[3, 1, 7], [9, 7, 0, 5, 3, 2], [1, 3, 4, 7]]  # 길이 3 · 6 · 4
lens = torch.tensor([len(s) for s in seqs])           # 진짜 길이표


# 왜 · 길이가 다른 줄을 같은 틀에 맞추고 길이표를 함께 만들기 위해
def pad(xs, T):                            # 길이 T 틀에 채우기
    X = torch.zeros(len(xs), T, 10)        # 빈 칸 = 열 칸 모두 0
    for b, s in enumerate(xs):             # 줄마다
        X[b, :len(s)] = to_tensor([s])[0]  # 앞쪽에 진짜 칸
    return X                               # 채운 텐서


# 왜 · 채운 텐서를 RNN에 넣어 마지막 칸이 진짜 끝이 아님을 확인하기 위해
X = pad(seqs, 6)                                                              # 가장 긴 6칸 틀
set_seed()                                                                    # 모델마다 같은 시작점
output, _ = nn.RNN(10, 32, batch_first=True)(X)                               # 채운 텐서를 그대로
idx = torch.arange(len(X))                                                    # 줄 번호 0 · 1 · 2
same = [torch.equal(a, b)                                                     # 줄마다 같은지
        for a, b in zip(output[:, -1], output[idx, lens - 1])]
print("X", tuple(X.shape), "· lens", lens.tolist(), "· -1 == lens-1:", same)  # 비교 결과
print("전체 시점 출력 → 칸마다 답:", tuple(nn.Linear(32, 2)(output).shape))   # 칸마다 점수

X (3, 6, 10) · lens [3, 6, 4] · -1 == lens-1: [False, True, False]
전체 시점 출력 → 칸마다 답: (3, 6, 2)


**실행 흐름**

1. 요약 꺼내는 방법 4가지를 고를 수 있는 `ReadNet` 정의
2. `forward()`에서 진짜 길이로 마지막 칸 · 평균 꺼내기

In [12]:
# 요약을 꺼내는 4가지 방법
# 왜 · 요약 꺼내는 방법만 다른 모델로 공정하게 비교하기 위해
class ReadNet(nn.Module):                          # 요약 꺼내는 방법만 다른 모델
    def __init__(self, how):                       # 꺼내는 방법
        super().__init__()                         # nn.Module 준비 (필수)
        self.r = nn.RNN(10, 32, batch_first=True)  # 같은 RNN
        self.o = nn.Linear(32, 2)                  # 같은 분류기
        self.how = how                             # 방법 기억

    # 왜 · 진짜 길이를 이용해 빈 칸을 피해 요약을 꺼내기 위해
    def forward(self, x, lens):                                   # 입력 + 길이표
        out, _ = self.r(x)                                        # (N, T, 32)
        if self.how == "마지막 칸 -1":                            # 틀의 끝
            h = out[:, -1]                                        # 마지막 칸 요약
        elif self.how == "실제 끝 lens-1":                        # 줄마다 진짜 끝
            h = out[torch.arange(len(x)), lens - 1]               # 줄마다 자기 끝
        elif self.how == "전부 평균":                             # 빈 칸까지 평균
            h = out.mean(1)                                       # 길이 축 평균
        else:                                                     # mask 평균: 진짜 칸만 1
            m = (torch.arange(x.shape[1], device=x.device)[None]  # 진짜 칸 = 1
                 < lens[:, None]).float().unsqueeze(-1)           # (N, T, 1)
            h = (out * m).sum(1) / m.sum(1)                       # 진짜 칸만 평균
        return self.o(h)                                          # 요약 → 점수

**실행 흐름**

1. 길이 5~60 줄 만들기 · 채우기
2. 4가지 방법을 같은 조건으로 학습
3. 방법별 정확도 출력

In [13]:
# 길이 5~60 인 줄 3000개로 4가지를 같은 조건에서 학습
# 왜 · 빈 칸을 읽는 버그가 점수에 드러나는지 같은 조건에서 확인하기 위해
def make_ragged(n, lo, hi, seed):     # 길이가 제각각인 줄
    rng = random.Random(seed)         # 이 데이터 전용 난수 (전역 seed와 분리)
    xs, ys = [], []                   # 입력 줄 · 정답을 모을 목록
    for _ in range(n):                # 줄 n개 만들기
        length = rng.randint(lo, hi)  # 줄마다 길이가 다름
        x, y = make_order_task(1, length, seed=rng.randrange(10**6))  # 줄 하나 만들기
        xs.append(x[0])               # 줄 저장
        ys.append(y[0])               # 정답 저장
    return xs, ys                     # 줄 목록 · 정답 목록


# 왜 · 학습 · 평가를 다른 seed로 만들어 겹치지 않게 하기 위해
xa, ya = make_ragged(3000, 5, 60, seed=11)                                # 학습 3000줄
xb, yb = make_ragged(1000, 5, 60, seed=12)                                # 평가 1000줄
T = max(len(s) for s in xa + xb)                                          # 가장 긴 길이 = 틀
Xa, Xb = pad(xa, T).to(DEV), pad(xb, T).to(DEV)                           # 채워서 GPU로
La = torch.tensor([len(s) for s in xa], device=DEV)                       # 학습 길이표
Lb = torch.tensor([len(s) for s in xb], device=DEV)                       # 평가 길이표
Ya, Yb = torch.tensor(ya, device=DEV), torch.tensor(yb, device=DEV)       # 정답
for how in ["마지막 칸 -1", "실제 끝 lens-1", "전부 평균", "mask 평균"]:  # 4가지 비교
    set_seed()                                                            # 모델마다 같은 시작점
    model = ReadNet(how).to(DEV)                                          # 방법마다 새 모델
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)                   # Adam
    for _ in range(25):                                                   # 25 epoch
        perm = torch.randperm(len(Xa)).to(DEV)                            # 줄 순서 섞기
        for i in range(0, len(Xa), 64):                                   # 64줄씩
            j = perm[i:i + 64]                                            # 이번 배치
            opt.zero_grad()                                               # 지난 기울기 지우기
            F.cross_entropy(model(Xa[j], La[j]), Ya[j]).backward()        # 손실 → 기울기
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)             # 기울기 폭발 방지
            opt.step()                                                    # 가중치 한 걸음 수정
    with torch.no_grad():                                                 # 평가는 기울기 기록 안 함
        acc = (model(Xb, Lb).argmax(1) == Yb).float().mean().item()       # 평가 정확도
    print(f"길이 5~60 · {kpad(how, 14)} → {acc:.3f}")                     # 방법별 결과

길이 5~60 · 마지막 칸 -1   → 0.500


길이 5~60 · 실제 끝 lens-1 → 1.000


길이 5~60 · 전부 평균      → 0.483


길이 5~60 · mask 평균      → 1.000


**결과** · 예측 정답은 `0.500` — 찍기. 빈 칸 요약에는 답이 남아 있지 않음

- `X (3, 6, 10) · lens [3, 6, 4] · -1 == lens-1: [False, True, False]` — 가장 긴 둘째 줄만 `-1`이 진짜 끝
- `전체 시점 출력 → 칸마다 답: (3, 6, 2)` — `out` 전체에 Linear → 칸마다 답 (빈 칸 답은 손실에서 빼야 함)
- `길이 5~60 · 마지막 칸 -1   → 0.500` — 빈 칸을 지난 요약을 읽음 → 찍기
- `길이 5~60 · 실제 끝 lens-1 → 1.000` — 줄마다 자기 끝을 읽으면 전부 맞힘
- `길이 5~60 · 전부 평균      → 0.483` — 빈 칸까지 평균 → 찍기
- `길이 5~60 · mask 평균    → 1.000` — 빈 칸을 빼고 평균해도 전부 맞힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>out[:, -1]</code> (마지막 시점)</td><td><strong>틀</strong>의 마지막 칸</td><td>모든 줄 길이가 같을 때 (HAR 128칸)</td></tr>
<tr><td><code>out[idx, lens - 1]</code></td><td>줄마다 <strong>진짜</strong> 마지막 칸</td><td>길이가 다른 줄, 한 줄에 답 하나</td></tr>
<tr><td><code>out</code> (전체 시점)</td><td>칸마다 요약 <code>(N, L, H)</code></td><td>칸마다 답이 필요할 때 (매 시점 예측 · 단어마다 태그)</td></tr>
<tr><td>mask 평균</td><td>진짜 칸만 평균</td><td>모든 칸을 고르게 반영할 때 (RNN-3 평균 풀링)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 길이 차이가 작으면(수업 코드의 4~12) 네 방법이 모두 100%라 버그가 드러나지 않음
> - 길이를 함께 들고 다니기

> ✅ **한 줄 정리** · 한 줄에 답 하나면 마지막 시점을 `lens - 1`로, 칸마다 답이면 전체 시점 `out`을 쓰고, 빈 칸은 항상 빼고 읽는다.

**딥러닝 코드에서는**

- RNN-3에서는 `pad_sequence`로 채우고 `lengths`를 함께 넘기며, `pack_padded_sequence`가 빈 칸을 아예 건너뜀

---
## 4. 멀리까지 닿는가

> **이 장의 흐름**
>
> - 3장까지는 길이 10 안팎의 짧은 줄이었다
> - 답을 정하는 칸과 끝이 멀어지면 학습 신호가 사라지는 문제와, 문(gate)을 단 LSTM · GRU

**파이프라인 위치** · ④ 모델 — 셀 종류 고르기 (RNN · LSTM · GRU)

### 4-1. `A` 거리와 기울기 소실 · `retain_grad()` <sub>수업 코드 L2 · L3 '기울기가 어디까지 오는가' · 강의</sub>

> 🎯 **답을 정하는 칸과 끝 사이가 멀어지면 RNN은 왜 갑자기 못 배우는가?**

💡 **쉽게 말하면**

- 역전파는 **귓속말 전달 게임을 거꾸로** 하는 것.
- 끝에서 '틀렸다'는 신호가 한 칸씩 되돌아가는데, 칸마다 약 절반만 남아 백 칸이면 사라짐.

<img src="images/rnn1_grad.png" alt="거슬러 간 칸별 기울기 크기" width="930">

**실행 흐름**

1. 3 · 7 을 맨 앞에 두고 거리 8 · 78 · 98 (20 epoch) · 98 (60 epoch)
2. `retain_grad`로 칸마다 기울기 크기 (float32 · float64)
3. 0.9 · 1.1을 50번 곱하기

**먼저 예측** · 거리 98에서 20 epoch로 학습하면 몇 %일까? — 답은 코드 실행 결과 아래에

In [14]:
# 3 · 7 을 맨 앞에 두고 끝까지의 거리를 늘리기 (같은 20 epoch)
# 왜 · 거리가 늘 때 RNN이 언제 무너지는지 같은 예산으로 확인하기 위해
def run_front(dist, epochs):                                   # 거리 · epoch로 학습
    xa, ya = make_front_task(4000, dist + 2, seed=1)           # 학습 4000줄
    xb, yb = make_front_task(1000, dist + 2, seed=2)           # 평가 1000줄
    set_seed()                                                 # 모델마다 같은 시작점
    return train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # 정확도 반환
                 to_tensor(xb), torch.tensor(yb), epochs=epochs)


# 왜 · 거리와 학습 길이를 바꿔 원인이 학습 부족인지 구분하기 위해
for dist, ep in [(8, 20), (78, 20), (98, 20), (98, 60)]:               # 거리 · epoch 4가지
    print(f"거리 {dist:>2} · {ep} epoch → {run_front(dist, ep):.3f}")  # 결과


# 기울기 크기 재기: 손으로 편 RNNCell + retain_grad (CPU)
# 왜 · 기울기가 멀리 거슬러 갈수록 작아지는지 직접 재기 위해
STEPS = (0, 10, 20, 40, 70, 99)  # 마지막 칸에서 거슬러 간 칸 수


# 왜 · 셀 종류 · 문 시작값 · 자료형만 바꿔 같은 방법으로 기울기를 재기 위해
def grad_profile(kind, T=100, hid=32, forget_bias=None, dtype=torch.float32):  # 칸별 기울기 크기
    set_seed()                                    # 모델마다 같은 시작점
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)  # 상자 하나
    head = nn.Linear(hid, 2).to(dtype)            # 끝 분류기
    if forget_bias is not None:                   # bias 순서 [input | forget | cell | output]
        with torch.no_grad():                     # 평가는 기울기 기록 안 함
            cell.bias_ih[hid:2 * hid].fill_(forget_bias)  # forget 구간에 값 넣기
            cell.bias_hh[hid:2 * hid].fill_(0.0)  # 두 bias가 더해지므로 한쪽은 0
    xs, ys = make_front_task(64, T, seed=7)       # 64줄 · 길이 T
    X = to_tensor(xs).to(dtype)                   # one-hot · 자료형 맞춤
    h = torch.zeros(64, hid, dtype=dtype)         # 처음 요약 0
    c = torch.zeros(64, hid, dtype=dtype)         # LSTM 메모 0
    hs = []                                       # 칸마다 요약을 모음
    for t in range(T):                            # T칸 차례로
        if kind == "RNN":                         # RNN은 h만
            h = cell(X[:, t], h)                  # 새 요약
        else:
            h, c = cell(X[:, t], (h, c))          # LSTM은 h · c 둘
        h.retain_grad()                           # 중간 요약의 기울기를 버리지 말고 남기기
        hs.append(h)                              # 이번 칸 요약 저장
    nn.CrossEntropyLoss()(head(hs[-1]), torch.tensor(ys)).backward()  # 끝에서 손실 → 역전파
    return [hs[T - 1 - k].grad.norm().item() for k in STEPS]  # k칸 거슬러 간 기울기 크기


# 왜 · 칸별 기울기를 같은 형식으로 나란히 출력하기 위해
def show(name, values):                                           # 한 줄 출력
    print(f"{name:<12}" + "".join(f"{v:>9.1e}" for v in values))  # 지수 표기 열 맞춤


print("거슬러 간 칸" + "".join(f"{k:>9}" for k in STEPS))            # 한글 폭 12칸
show("RNN f32", grad_profile("RNN"))                                 # float32
show("RNN f64", grad_profile("RNN", dtype=torch.float64))            # float64로 같은 계산
print(f"0.9 ** 50 = {0.9 ** 50:.3f} · 1.1 ** 50 = {1.1 ** 50:.0f}")  # 반복 곱 감 잡기

거리  8 · 20 epoch → 1.000


거리 78 · 20 epoch → 1.000


거리 98 · 20 epoch → 0.517


거리 98 · 60 epoch → 1.000
거슬러 간 칸        0       10       20       40       70       99
RNN f32       5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
RNN f64       5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  3.4e-28
0.9 ** 50 = 0.005 · 1.1 ** 50 = 117


**결과** · 예측 정답은 `0.517` — 78까지는 100%였는데 98에서 갑자기 찍기 (60 epoch로 늘리면 `1.000`)

- `거리 78 · 20 epoch → 1.000` — 78칸까지는 20 epoch로 닿음 (거리 8도 `1.000`)
- `거리 98 · 20 epoch → 0.517` — 98칸에서 갑자기 찍기 수준 — 절벽
- `거리 98 · 60 epoch → 1.000` — 세 배 학습하면 닿음 → 기억을 못 하는 게 아니라 배우기가 느림
- `RNN f32      5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00` — 거슬러 갈수록 기울기가 급격히 작아짐 (20칸에서 10만 배 넘게)
- `RNN f64      5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  3.4e-28` — 99칸은 진짜 0이 아니라 float32가 못 적는 크기 (학습에는 0과 같음)
- `0.9 ** 50 = 0.005 · 1.1 ** 50 = 117` — 1보다 조금 작은 비율을 반복해 곱하면 소실, 조금 크면 폭발

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>기울기 소실</td><td>1보다 작은 비율이 반복해 곱해져 0으로</td><td>먼 과거를 고칠 신호가 안 옴 → LSTM · GRU (4-2)</td></tr>
<tr><td>기울기 폭발</td><td>1보다 큰 비율이 반복해 곱해져 커짐 → <code>nan</code></td><td><code>clip_grad_norm_</code>으로 잘라냄 (5-1 학습 함수)</td></tr>
<tr><td>epoch 늘리기</td><td>느리게라도 배울 시간을 줌</td><td>거리 98처럼 경계에 있을 때</td></tr>
<tr><td><code>output.grad</code>로 재기</td><td>h의 기울기가 아님 (계산 길이 다름)</td><td>쓰지 않음 → 손으로 편 <code>RNNCell</code> + <code>retain_grad</code></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `nn.RNN`의 `output.grad`는 칸 사이로 넘어가는 `h`의 기울기가 아님
> - 측정은 손으로 편 `RNNCell` + `h.retain_grad()`

> ✅ **한 줄 정리** · 기울기는 칸을 하나씩 거슬러 가며 매번 비슷한 비율로 곱해져 먼 칸일수록 사라지므로, RNN은 먼 과거를 배우기가 매우 느리다.

**딥러닝 코드에서는**

- 학습 루프의 `nn.utils.clip_grad_norm_(model.parameters(), 5.0)`은 폭발 쪽 대처, 소실 쪽은 LSTM
- GRU로 구조를 바꿈

### 4-2. `A` LSTM과 forget 문 · `nn.LSTM` <sub>수업 코드 L3b '문을 달면 살아날까' · 레이어 실습 LSTM</sub>

> 🎯 **LSTM은 무엇을 더 돌려주며, 기울기 소실을 어떻게 늦추는가?**

💡 **쉽게 말하면**

- LSTM은 **회의 노트(c)와 지금 하는 말(h)을 따로** 둠.
- 노트는 지우기(×) · 덧붙이기(+)만 하고, 문(0~1)으로 얼마나 남길지 정함 — 문이 열려 있으면 신호가 곧게 지나감.

<img src="images/rnn1_lstm_cell.png" alt="LSTM 한 칸" width="703">

**LSTM 한 칸 계산표** (PyTorch `nn.LSTM` 문서 표기, σ = sigmoid 0~1, ⊙ = 같은 자리끼리 곱)

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">기호</th><th style="text-align: left;">이름</th><th style="text-align: left;">식</th><th style="text-align: left;">하는 일</th></tr>
</thead>
<tbody>
<tr><td><code>f</code></td><td>forget 문</td><td><code>σ(W·x + U·h + b)</code></td><td>메모에서 <strong>남길</strong> 양</td></tr>
<tr><td><code>i</code></td><td>input 문</td><td><code>σ(W·x + U·h + b)</code></td><td>메모에 <strong>적을</strong> 양</td></tr>
<tr><td><code>g</code></td><td>적을 내용</td><td><code>tanh(W·x + U·h + b)</code></td><td>새로 적을 내용</td></tr>
<tr><td><code>o</code></td><td>output 문</td><td><code>σ(W·x + U·h + b)</code></td><td>메모에서 <strong>꺼낼</strong> 양</td></tr>
<tr><td><code>c</code></td><td>메모 (cell state)</td><td><code>f ⊙ c + i ⊙ g</code></td><td>곱하기 · 더하기만 (통째로 다시 쓰지 않음)</td></tr>
<tr><td><code>h</code></td><td>꺼낸 말 (hidden state)</td><td><code>o ⊙ tanh(c)</code></td><td>다음 칸 · 출력으로 나감</td></tr>
</tbody></table>

네 문의 bias는 `[input | forget | cell | output]` 순서로 한 텐서에 붙어 있어 forget 문 bias는 `bias_ih[H:2*H]`

**실행 흐름**

1. `nn.LSTM`의 `output, (h_n, c_n)` 모양 · 값 비교
2. 같은 조건에서 RNN · LSTM(forget bias 0 · 2 · 4) 기울기
3. 문 시작값

**먼저 예측** · forget 문 bias를 0으로 둔 보통 LSTM은 99칸 뒤에 기울기가 남을까? — 답은 코드 실행 결과 아래에

In [15]:
# LSTM 이 돌려주는 것: output, (h_n, c_n)
# 왜 · LSTM 출력 모양을 RNN과 비교해 확인하기 위해
set_seed()                                                             # 모델마다 같은 시작점
lstm = nn.LSTM(10, 32, batch_first=True)                               # 입력 10 · hidden 32
x = to_tensor(xtr[:8])                                                 # (8, 10, 10)
output, (h_n, c_n) = lstm(x)                                           # 괄호 안 두 개 = 튜플 풀기
print("output", tuple(output.shape), "· h_n", tuple(h_n.shape),        # 세 모양
      "· c_n", tuple(c_n.shape))
print("output[:, -1] == h_n[0]:", torch.equal(output[:, -1], h_n[0]),  # h_n = 마지막 칸
      "| h_n == c_n:", torch.equal(h_n, c_n))


# 같은 조건에서 기울기 비교: forget 문 bias 0 · 2 · 4
# 왜 · forget 문 시작값만 바꿔 기울기가 멀리 오는지 비교하기 위해
print("거슬러 간 칸" + "".join(f"{k:>9}" for k in STEPS))           # 한글 폭 12칸
show("RNN", grad_profile("RNN"))                                    # 비교 기준
for fb in [0.0, 2.0, 4.0]:                                          # forget bias 3가지
    show(f"LSTM f+{fb:.0f}", grad_profile("LSTM", forget_bias=fb))  # LSTM 기울기
gate = torch.sigmoid(torch.tensor([0.0, 2.0, 4.0]))                 # bias → 문 시작값
print("forget 문 시작값 sigmoid(0 · 2 · 4):",                       # 0.5 · 0.88 · 0.98
      " · ".join(f"{v:.2f}" for v in gate.tolist()))

output (8, 10, 32) · h_n (1, 8, 32) · c_n (1, 8, 32)
output[:, -1] == h_n[0]: True | h_n == c_n: False
거슬러 간 칸        0       10       20       40       70       99
RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
LSTM f+0      5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00
LSTM f+2      5.3e-02  1.0e-03  5.2e-04  1.9e-04  3.9e-05  1.6e-05


LSTM f+4      5.3e-02  7.2e-04  5.9e-04  6.7e-04  9.8e-04  4.5e-03
forget 문 시작값 sigmoid(0 · 2 · 4): 0.50 · 0.88 · 0.98


**결과** · 예측 정답은 안 남음 — `0.0e+00`. 문이 반쯤(0.5) 열린 채 100번 지나면 RNN처럼 사라짐

- `output (8, 10, 32) · h_n (1, 8, 32) · c_n (1, 8, 32)` — LSTM은 돌려주는 것이 셋 (`h_n` · `c_n` 모양은 같음)
- `output[:, -1] == h_n[0]: True | h_n == c_n: False` — `h_n`은 output 마지막 칸, `c_n`(메모)은 바깥에 안 나오는 다른 값
- `LSTM f+0     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00` — 문이 0.5에서 출발 → RNN처럼 사라짐 (10칸에서는 RNN보다 작음)
- `LSTM f+2     5.3e-02  1.0e-03  5.2e-04  1.9e-04  3.9e-05  1.6e-05` — 문을 열어 두면(0.88) 99칸 뒤에도 남음
- `LSTM f+4     5.3e-02  7.2e-04  5.9e-04  6.7e-04  9.8e-04  4.5e-03` — 더 열면(0.98) 멀어져도 거의 줄지 않음
- `forget 문 시작값 sigmoid(0 · 2 · 4): 0.50 · 0.88 · 0.98` — bias를 더하면 문이 열린 상태로 학습을 시작

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>output</code></td><td>모든 칸의 <code>h</code> → <code>(N, L, H)</code></td><td>칸마다 답 · 평균 풀링</td></tr>
<tr><td><code>h_n</code></td><td>마지막 칸의 <code>h</code>, <strong>층마다</strong> → <code>(층, N, H)</code></td><td>한 줄에 답 하나 (마지막 층 <code>h_n[-1]</code>)</td></tr>
<tr><td><code>c_n</code></td><td>LSTM의 메모(cell state), 바깥 출력이 아님</td><td>다음 구간으로 상태를 이어 줄 때만 (<code>h_n</code>과 함께)</td></tr>
<tr><td>forget bias 키우기</td><td>문이 열린 채로 학습 시작</td><td>아주 먼 과거까지 이어야 할 때 시작값 조정</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 'LSTM이 기울기 소실을 해결했다'는 부정확
> - 문이 열려 있을 때 크게 늦출 뿐이고, 여는 정도를 학습으로 정하게 한 것

> ✅ **한 줄 정리** · LSTM은 `output, (h_n, c_n)`을 돌려주며, 메모 `c`를 문으로 남겨 두는 덧셈 길 덕분에 문이 열려 있으면 기울기가 멀리까지 간다.

**딥러닝 코드에서는**

- `out, (h, c) = self.lstm(x)`
- 분류에는 `h[-1]`, `c`는 다음 구간으로 상태를 이어 줄 때만 `(h, c)` 그대로 넘김

### 4-3. `A` GRU와 파라미터 수 · `nn.GRU` <sub>수업 코드 L4 '파라미터 수를 비슷하게' · 레이어 실습 GRU · 강의</sub>

> 🎯 **RNN · LSTM · GRU는 같은 hidden 크기에서 파라미터가 왜 다르고, 어떻게 손으로 세는가?**

💡 **쉽게 말하면**

- 같은 hidden 64라도 안에서 계산하는 **벌 수**가 다름.
- RNN 1벌, GRU 3벌(문 2 + 후보 1), LSTM 4벌(문 3 + 내용 1).

**실행 흐름**

1. GRU의 반환값 모양
2. 벌 수 × (H × (I + H) + 2H) 손계산
3. 코드 파라미터 수 · `weight_ih` 모양과 비교
4. 분류기 Linear 더하기

**먼저 예측** · 입력 9 · hidden 64에서 LSTM 층의 파라미터는 RNN의 몇 배일까? — 답은 코드 실행 결과 아래에

In [16]:
# GRU 도 RNN 처럼 (out, h_n) 두 개만 돌려줌
# 왜 · GRU의 출력 모양을 LSTM과 비교하기 위해
gru = nn.GRU(10, 32, batch_first=True)                         # GRU 층
out, h_n = gru(to_tensor(xtr[:8]))                             # RNN처럼 두 개
print("GRU out", tuple(out.shape), "· h_n", tuple(h_n.shape))  # 모양


# 파라미터 손계산: k벌 × (H × (I + H) + 2H) · RNN 1벌 · GRU 3벌 · LSTM 4벌
# 왜 · 셀 종류마다 파라미터가 몇 벌인지 공식으로 세어 비용을 비교하기 위해
I, H = 9, 64                                           # 5장 HAR 크기: 입력 9 · hidden 64
for kind, k in [("RNN", 1), ("GRU", 3), ("LSTM", 4)]:  # 셀 · 벌 수
    layer = getattr(nn, kind)(I, H, batch_first=True)  # 층 만들기
    hand = k * (H * (I + H) + 2 * H)                   # 손계산
    print(f"{kind:<4} 손계산 {hand:>6,} · 코드 {n_params(layer):>6,}"  # 손계산 · 코드 비교
          f" · weight_ih {tuple(layer.weight_ih_l0.shape)}"
          f" · + Linear(64→6) = {hand + H * 6 + 6:,}")

GRU out (8, 10, 32) · h_n (1, 8, 32)
RNN  손계산  4,800 · 코드  4,800 · weight_ih (64, 9) · + Linear(64→6) = 5,190
GRU  손계산 14,400 · 코드 14,400 · weight_ih (192, 9) · + Linear(64→6) = 14,790
LSTM 손계산 19,200 · 코드 19,200 · weight_ih (256, 9) · + Linear(64→6) = 19,590


**결과** · 예측 정답은 4배 — 19,200 ÷ 4,800

- `GRU out (8, 10, 32) · h_n (1, 8, 32)` — GRU는 메모(`c_n`)가 없어 RNN과 같은 두 개
- `RNN  손계산  4,800 · 코드  4,800 · weight_ih (64, 9) · + Linear(64→6) = 5,190` — 1벌 (64 × 73 + 128 = 4800)
- `GRU  손계산 14,400 · 코드 14,400 · weight_ih (192, 9) · + Linear(64→6) = 14,790` — 3벌 → `weight_ih` 행이 3 × 64 = 192
- `LSTM 손계산 19,200 · 코드 19,200 · weight_ih (256, 9) · + Linear(64→6) = 19,590` — 4벌 → 4 × 64 = 256, 같은 hidden 64라도 RNN의 4배

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>RNN</td><td>문 없음 · <strong>1벌</strong> · 넘기는 줄 1개(<code>h</code>)</td><td>짧은 순서, 가중치가 가장 적음 → <strong>먼저 시도</strong></td></tr>
<tr><td>LSTM</td><td>문 3개 (forget · input · output) · <strong>4벌</strong> · 줄 2개(<code>h</code>, <code>c</code>)</td><td>긴 의존, 메모를 따로 지켜야 할 때</td></tr>
<tr><td>GRU</td><td>문 2개 (reset · update) · <strong>3벌</strong> · 줄 1개</td><td>LSTM과 비슷한 성능을 더 적은 파라미터로</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - hidden 크기가 같다고 공정한 비교가 아님
> - 크기 차이인지 구조 차이인지 가르려면 파라미터 수도 맞춰 봄 (6장)

> ✅ **한 줄 정리** · 파라미터 수는 `벌 수 × (H × (I + H) + 2H)`이고 RNN 1벌 · GRU 3벌 · LSTM 4벌이라, 같은 hidden이면 GRU가 LSTM의 3/4이다.

**딥러닝 코드에서는**

- `sum(p.numel() for p in model.parameters())`로 모델 크기를 항상 출력
- 프로젝트 보고서의 모델 비교표 필수 칸

---
## 5. 실제 데이터

> **이 장의 흐름**
>
> - 4장까지는 정답 규칙을 아는 만든 문제였다
> - 아무도 100%를 못 받는 실제 센서 데이터로 옮겨 6장 비교의 시험대를 준비

**파이프라인 위치** · ① 데이터 준비 · ② 텐서로 · ③ 전처리 — 내려받기, `(N, 128, 9)`, train 통계 정규화

### 5-1. `A` UCI HAR 불러오기와 정규화 · `mean((0, 1))` <sub>수업 코드 'UCI HAR 열어보기' · 강의</sub>

> 🎯 **실제 센서 기록을 RNN 입력 `(N, 128, 9)`로 만들고, 정규화 통계는 어디서 계산하는가?**

💡 **쉽게 말하면**

- 폰을 허리에 차고 움직인 기록을 **2.56초(128칸)씩 자른 것**.
- 한 칸 = 가속도 · 자이로 · 전체 가속도 × xyz = 숫자 9개.
- 정규화 기준은 시험지(test)를 보기 전에 정한 train 값만.

<img src="images/rnn1_har.png" alt="UCI HAR 한 줄" width="895">

**실행 흐름**

1. 내려받기(처음 한 번)
2. 9채널 파일 읽기
3. `permute`로 `(N, 128, 9)`
4. 라벨 개수
5. train 통계로 정규화
6. 모델 · 학습 함수 정의

**먼저 예측** · train 통계로 정규화하면 test의 평균도 정확히 0이 될까? — 답은 코드 실행 결과 아래에

In [17]:
# UCI HAR 내려받기 (처음 한 번만, 약 60MB · 노트북 옆 UCI_HAR/ 폴더)
# 왜 · 인터넷은 처음 한 번만 쓰고 그다음부터는 노트북 옆 폴더의 파일을 쓰기 위해
URL = ("https://archive.ics.uci.edu/static/public/240/"             # 수업 코드의 주소
       "human+activity+recognition+using+smartphones.zip")
DATA = "UCI_HAR"                                                    # 노트북 옆 폴더
ROOT = os.path.join(DATA, "UCI HAR Dataset")                        # 압축 푼 폴더
if not os.path.exists(ROOT):                                        # 처음 한 번만
    os.makedirs(DATA, exist_ok=True)                                # 받을 폴더
    urllib.request.urlretrieve(URL, os.path.join(DATA, "har.zip"))  # 약 60MB 내려받기
    zipfile.ZipFile(os.path.join(DATA, "har.zip")).extract(         # 바깥 압축
        "UCI HAR Dataset.zip", DATA)                                # 압축 안에 압축이 하나 더
    zipfile.ZipFile(os.path.join(DATA, "UCI HAR Dataset.zip")).extractall(DATA)  # 안쪽 압축


# 센서 9채널 × 128칸 읽기 → (줄 수, 128, 9)
# 왜 · 센서 파일을 RNN이 받는 (줄 수, 길이, 채널) 모양으로 읽기 위해
CH = ["body_acc_x", "body_acc_y", "body_acc_z",  # 센서 9채널 파일 이름
      "body_gyro_x", "body_gyro_y", "body_gyro_z",
      "total_acc_x", "total_acc_y", "total_acc_z"]


# 왜 · 학습 · 평가 데이터를 같은 방법으로 읽기 위해
def load_har(split):                                                          # train · test 읽기
    chans = []                                                                # 채널마다 표
    for c in CH:                                                              # 9채널
        name = f"{c}_{split}.txt"                                             # 파일 이름
        with open(os.path.join(ROOT, split, "Inertial Signals", name)) as f:  # 채널 파일 열기
            chans.append([[float(v) for v in line.split()] for line in f])    # 줄마다 128개 숫자
    # (9, N, 128) → (N, 128, 9): 채널 축을 맨 뒤로
    X = torch.tensor(chans).permute(1, 2, 0).contiguous()
    with open(os.path.join(ROOT, split, f"y_{split}.txt")) as f:              # 정답 파일
        y = torch.tensor([int(v) - 1 for v in f.read().split()])              # 1~6 → 0~5
    return X, y                                                               # 입력 · 정답


# 왜 · 학습 · 평가 데이터를 한 번만 읽어 두고 뒤에서 다시 쓰기 위해
Xh, Yh = load_har("train")                                     # 학습 7352줄
Xh_te, Yh_te = load_har("test")                                # 평가 2947줄
print("train", tuple(Xh.shape), "· test", tuple(Xh_te.shape))  # 모양
print("train 라벨 0~5 개수:", torch.bincount(Yh).tolist())     # 활동별 개수

train (7352, 128, 9) · test (2947, 128, 9)
train 라벨 0~5 개수: [1226, 1073, 986, 1286, 1374, 1407]


**실행 흐름**

1. train 통계로 정규화
2. HAR 모델 · 학습 함수 정의

In [18]:
# 정규화: train 의 채널별 평균 · 표준편차로만 (test 에도 같은 값 사용)
# 왜 · 평가 데이터 정보가 새지 않게 train 통계로만 정규화하기 위해
mu = Xh.mean((0, 1), keepdim=True)                                        # (1, 1, 9)
sd = Xh.std((0, 1), keepdim=True)                                         # train 채널별 표준편차
Xn, Xn_te = (Xh - mu) / sd, (Xh_te - mu) / sd                             # test도 train 값으로
print(f"정규화 후 train 평균 {Xn.mean():.3f} · 표준편차 {Xn.std():.3f}")  # 0 · 1 확인
print(f"정규화 후 test  평균 {Xn_te.mean():.3f} · 표준편차 {Xn_te.std():.3f}")  # 조금 벗어남


# HAR 모델 · 학습 함수 (6 · 7장에서 사용)
# 왜 · 셀 종류 · 크기만 바꿔 쓸 수 있는 HAR 모델을 만들기 위해
class HarNet(nn.Module):                                               # HAR 분류 모델
    def __init__(self, kind, hid):                                     # 셀 종류 · hidden
        super().__init__()                                             # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(9, hid, batch_first=True)                       # 입력 9채널
        self.o = nn.Linear(hid, 6)                                     # 활동 6가지

    def forward(self, x):             # 입력이 지나가는 길
        out, _ = self.r(x)            # 모든 칸 요약
        return self.o(out[:, -1, :])  # 마지막 칸 → 6점수


# 왜 · 셀 종류 · 크기 · seed만 바꾸고 나머지는 같은 조건으로 학습하기 위해
def run_har(kind, hid, seed=SEED, Xa=None, Xb=None, epochs=12):  # 학습 → 정확도 · 파라미터 · 시간
    Xa = (Xn if Xa is None else Xa).to(DEV)                      # 기본은 정규화한 train
    Xb = (Xn_te if Xb is None else Xb).to(DEV)                   # 기본은 정규화한 test
    Ya, Yb = Yh.to(DEV), Yh_te.to(DEV)                           # 정답도 GPU로
    set_seed(seed)                                               # seed마다 다른 시작점
    m = HarNet(kind, hid).to(DEV)                                # 새 모델
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)              # Adam 1e-3
    loss_fn = nn.CrossEntropyLoss()                              # 분류 손실
    t = time.time()                                              # 시간 재기 시작
    for _ in range(epochs):                                      # epoch 반복
        perm = torch.randperm(len(Xa)).to(DEV)                   # 줄 순서 섞기
        for i in range(0, len(Xa), 64):                          # 64줄씩
            j = perm[i:i + 64]                                   # 이번 배치
            opt.zero_grad()                                      # 지난 기울기 지우기
            loss_fn(m(Xa[j]), Ya[j]).backward()                  # 손실 → 기울기
            nn.utils.clip_grad_norm_(m.parameters(), 5.0)        # 기울기 폭발 방지
            opt.step()                                           # 가중치 한 걸음 수정
    m.eval()                                                     # 평가 모드
    with torch.no_grad():                                        # 평가는 기울기 기록 안 함
        acc = (m(Xb).argmax(1) == Yb).float().mean().item()      # test 정확도
    return round(acc, 4), n_params(m), time.time() - t           # 결과 셋

정규화 후 train 평균 0.000 · 표준편차 1.000
정규화 후 test  평균 -0.007 · 표준편차 0.953


**결과** · 예측 정답은 아님 — test는 `-0.007` · 표준편차 `0.953`. train 값으로 맞췄으니 train만 정확히 0 · 1

- `train (7352, 128, 9) · test (2947, 128, 9)` — 줄 수 · 길이 128칸(2.56초) · 센서 9채널 — 3장의 세 겹 그대로
- `train 라벨 0~5 개수: [1226, 1073, 986, 1286, 1374, 1407]` — 활동 6가지가 비교적 고르게 있음 (찍기는 약 17%)
- `정규화 후 train 평균 0.000 · 표준편차 1.000` — train 통계로 맞췄으니 정확히 0 · 1
- `정규화 후 test  평균 -0.007 · 표준편차 0.953` — test는 train 통계를 그대로 써서 0 · 1에서 조금 벗어남 (정상)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>train 통계로 정규화</td><td>train의 평균 · 표준편차만 사용</td><td><strong>항상</strong> (test 정보가 새지 않음)</td></tr>
<tr><td>전체 통계로 정규화</td><td>test 값이 평균에 섞임</td><td>쓰지 않음 (평가가 실제보다 좋게 나옴)</td></tr>
<tr><td>채널별 <code>mean((0, 1))</code></td><td>채널 9개 각각의 평균 · 표준편차</td><td>센서마다 단위 · 크기가 다를 때</td></tr>
<tr><td>전체 하나 <code>mean()</code></td><td>모든 값을 하나의 평균으로</td><td>모든 채널 단위가 같을 때만</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `X_train.txt`(561개 요약 특징)는 순서가 없는 표 데이터
> - RNN에는 `Inertial Signals` 9채널을 씀

> ✅ **한 줄 정리** · 센서 기록은 `(줄 수, 길이, 채널)`로 쌓고, 정규화 평균 · 표준편차는 train에서만 계산해 test에 그대로 쓴다.

**딥러닝 코드에서는**

- `mu, sd = X_train.mean((0, 1)), X_train.std((0, 1))` → 저장해 두고 추론 때도 같은 값 사용 (RNN-2 시계열에서 다시 강조)

---
## 6. 종합 실습 · UCI HAR로 RNN · LSTM · GRU 같은 조건 비교

> **이 장의 흐름**
>
> - 1 ~ 5장을 한 번에 사용
> - 수업 L4 '실제 데이터에서 비교'와 같은 조건으로, 한 번에 하나만 바꿔 잼

**파이프라인 위치** · ④ 모델 → ⑤ 학습 → ⑥ 평가

> 🎯 **같은 조건에서 RNN · LSTM · GRU를 학습하면 정확도와 파라미터 수는 어떻게 다르고, 무엇으로 골라야 하는가?**

💡 **쉽게 말하면**

- 같은 시험지 · 같은 공부 시간 · 같은 교재로 세 학생을 비교.
- 먼저 **같은 hidden 64**(구조만 바꾼 비교), 다음은 **파라미터 수를 맞춘 비교**(RNN은 128, GRU는 74로 키움).

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 것</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>hidden 64로 세 모델 학습 · 평가</td><td>5-1 <code>run_har()</code> · 0장 <code>set_seed()</code></td></tr>
<tr><td>2</td><td>정확도 · 파라미터 수 표</td><td>4-3 파라미터 손계산</td></tr>
<tr><td>3</td><td>RNN 128 · GRU 74로 파라미터를 LSTM(19,590) 근처로 맞춰 다시</td><td>4-3</td></tr>
<tr><td>4</td><td>학습 시간 (환경마다 다름)</td><td><code>time.time()</code></td></tr>
</tbody></table>

**먼저 예측** · hidden 64에서 가장 높은 모델은? 파라미터를 맞추면 RNN은 LSTM · GRU를 따라잡을까? — 답은 코드 실행 결과 아래에

**실행 흐름**

1. 같은 조건에서 RNN · LSTM · GRU 학습 · 정확도 출력
2. 파라미터 수를 맞춘 비교

In [19]:
# 같은 조건: hidden 64 · Adam 1e-3 · batch 64 · 12 epoch · 같은 seed
# 왜 · 셀 종류만 바꾼 공정한 조건에서 실제 데이터 성능을 비교하기 위해
sec = {}                                             # 모델별 학습 시간
print("모델  hidden  정확도  파라미터")              # 한글은 2칸 폭 → 아래 줄과 열 맞춤
for kind in ["RNN", "LSTM", "GRU"]:                  # 세 모델
    acc, n, sec[kind] = run_har(kind, 64)            # 같은 hidden 64
    print(f"{kind:<5} {64:>6}  {acc:.4f}  {n:>8,}")  # 한 줄 결과


# 조건을 맞춘 비교: 파라미터 수를 LSTM(19,590) 근처로
# 왜 · 크기 차이 때문인지 구조 차이 때문인지 나눠 보기 위해
for kind, hid in [("RNN", 128), ("GRU", 74)]:         # 파라미터를 LSTM 근처로
    acc, n, _ = run_har(kind, hid)                    # 같은 조건으로 학습
    print(f"{kind:<5} {hid:>6}  {acc:.4f}  {n:>8,}")  # 한 줄 결과

모델  hidden  정확도  파라미터


RNN       64  0.8073     5,190


LSTM      64  0.8683    19,590


GRU       64  0.9026    14,790


RNN      128  0.8554    18,566


GRU       74  0.9097    19,320


**실행 흐름**

1. 모델별 학습 시간(초) 출력 · 환경마다 다름

In [20]:
# 학습 시간 (초) · 환경마다 다름
# 왜 · 비용 비교의 참고로 시간을 기록하되 환경마다 다르다는 점을 밝히기 위해
print("시간(초):", " · ".join(f"{k} {v:.1f}" for k, v in sec.items()))  # 환경마다 다름

시간(초): RNN 9.6 · LSTM 10.2 · GRU 8.4


**결과** · 예측 정답은 이번 실행에서 GRU(`0.9026`)가 가장 높음 — RNN은 파라미터를 3.5배 넘게 늘려도(`18,566`) `0.8554`로 GRU 64와 4.7%p 차이

- `RNN       64  0.8073     5,190` — 같은 hidden 64에서 RNN 80.7%
- `LSTM      64  0.8683    19,590` — LSTM 86.8%, 파라미터는 RNN의 약 3.8배
- `GRU       64  0.9026    14,790` — GRU 90.3%, LSTM보다 파라미터가 25% 적음
- `RNN      128  0.8554    18,566` — 파라미터를 LSTM 수준으로 키운 RNN도 85.5% → 크기만으로는 못 따라잡음
- `GRU       74  0.9097    19,320` — GRU를 LSTM 크기로 맞추면 91.0%

<img src="images/rnn1_compare.png" alt="같은 조건 비교 막대" width="721">

**해석**

- 구조만 바꾼 비교와 크기를 맞춘 비교에서 **같은 모양**이 나옴
- 문이 없는 RNN은 크기를 키워도 아래, 문이 있는 두 모델이 위
- 강의 화면(CPU)은 RNN 82.5% · LSTM 90.6% · GRU 90.1%로 LSTM이 위
- 7-2에서 seed 5개로 재면 LSTM과 GRU 차이는 흔들림과 비슷한 크기 → **성능이 아니라 비용**(파라미터 · 시간)으로 고름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>hidden을 같게 (구조만 바꾼 비교)</td><td>같은 64라도 벌 수가 달라 파라미터 1 : 3 : 4</td><td>구조 자체의 차이를 먼저 볼 때</td></tr>
<tr><td>파라미터를 같게 (조건을 맞춘 비교)</td><td>적은 쪽의 hidden을 키움</td><td>'파라미터가 많아서 이겼다'는 반론을 막을 때</td></tr>
<tr><td>RNN</td><td>가중치가 가장 적음, 이 데이터에서 가장 낮음</td><td><strong>먼저</strong> 돌려 기준으로 삼기</td></tr>
<tr><td>LSTM · GRU</td><td>문이 있어 긴 의존에 강함, 두 모델 성능은 비슷</td><td>RNN이 부족할 때, 여유가 없으면 GRU부터</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한 번 실행한 정확도 차이로 모델 순위를 정하지 않기
> - 7-2처럼 여러 seed 평균 · 폭과 함께 비교

> ✅ **한 줄 정리** · 같은 조건에서 하나만 바꿔 재고, 조건을 맞춰 한 번 더 재서 같은 모양이면 그 차이를 믿고, 비슷하면 비용으로 고른다.

**딥러닝 코드에서는**

- 프로젝트에서 시계열 · 텍스트 모델을 고를 때도 같은 순서
- RNN(또는 기준선) → LSTM · GRU 같은 조건 비교 → 파라미터 · 시간 · seed 흔들림을 표 하나에 정리

---
## 7. 수업 과제 빈칸 정답 · 검증

> **이 장의 흐름**
>
> - 6장은 hidden 64 · seed 하나의 비교였다
> - 수업 코드 과제 2 · 3 · 4의 빈칸을 채워 크기 · seed · 순서를 바꿔도 결론이 유지되는지 확인 (과제 1은 6장 해석, 과제 5는 11장 아홉 단계)

**파이프라인 위치** · ⑥ 평가 — 결론의 흔들림 확인

### 7-1. `B` 과제 2 hidden 크기 바꾸기 · `hidden_size` <sub>수업 코드 '과제 2' 빈칸 2곳</sub>

> 🎯 **hidden 크기를 32 · 128로 바꾸면 파라미터 수와 정확도는 어떻게 움직이는가?**

💡 **쉽게 말하면**

- hidden은 **메모장 칸 수**.
- 칸을 늘리면 적을 수 있는 것은 많아지지만 파라미터는 대략 제곱으로 늘어남.

**실행 흐름**

1. 빈칸 `(___, ___)` · `(32, 128)`
2. 세 모델 × 두 크기 학습
3. 정확도 · 파라미터 표

**먼저 예측** · RNN을 128로 키우면 LSTM · GRU 32를 넘을까? — 답은 코드 실행 결과 아래에

In [21]:
# 과제 2 빈칸 정답: (___, ___) → (32, 128)
# 왜 · 크기만 바꿨을 때 정확도 · 파라미터가 어떻게 바뀌는지 확인하기 위해
for kind in ["RNN", "LSTM", "GRU"]:     # 세 모델
    for hid in (32, 128):               # 빈칸 정답
        acc, n, _ = run_har(kind, hid)  # 같은 조건으로 학습
        print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,}")  # 결과

RNN   hidden 32   정확도 0.7635 · 파라미터  1,574


RNN   hidden 128  정확도 0.8554 · 파라미터 18,566


LSTM  hidden 32   정확도 0.9040 · 파라미터  5,702


LSTM  hidden 128  정확도 0.9077 · 파라미터 71,942


GRU   hidden 32   정확도 0.8951 · 파라미터  4,326


GRU   hidden 128  정확도 0.9063 · 파라미터 54,150


**결과** · 예측 정답은 못 넘음 — RNN 128 `0.8554` < LSTM 32 `0.9040` · GRU 32 `0.8951`

- `RNN   hidden 32   정확도 0.7635 · 파라미터  1,574` — 작은 RNN은 76.4%
- `RNN   hidden 128  정확도 0.8554 · 파라미터 18,566` — 키우면 85.5%지만 LSTM · GRU의 32보다 낮음
- `LSTM  hidden 32   정확도 0.9040 · 파라미터  5,702` — 이번 실행에서는 LSTM 64(`0.8683`)보다 높음 → 한 번 실행 값은 흔들림
- `LSTM  hidden 128  정확도 0.9077 · 파라미터 71,942` — 파라미터가 약 12.6배인데 32와 0.4%p 차이
- `GRU   hidden 32   정확도 0.8951 · 파라미터  4,326` — GRU 32도 이미 89.5%
- `GRU   hidden 128  정확도 0.9063 · 파라미터 54,150` — 키워도 1.1%p 오름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>hidden 작게 (32)</td><td>파라미터 적음 · 계산량 적음</td><td>데이터가 적을 때, 처음 시도</td></tr>
<tr><td>hidden 크게 (128)</td><td>파라미터가 약 4배 이상</td><td>작은 모델이 확실히 부족할 때</td></tr>
<tr><td>구조 바꾸기 (RNN → LSTM · GRU)</td><td>문을 달아 먼 과거를 잇기 쉬움</td><td>크기를 키워도 안 오를 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 크기를 키운 효과가 seed 흔들림(7-2)보다 작으면 '커서 좋아졌다'고 말할 수 없음

> ✅ **한 줄 정리** · 이 데이터에서는 hidden을 키우는 것보다 문이 있는 구조(LSTM · GRU)로 바꾸는 효과가 훨씬 크다.

**딥러닝 코드에서는**

- 모델 크기는 작은 것부터 시도하고, 검증 정확도가 멈추면 구조를 바꾸거나 데이터를 늘림

### 7-2. `B` 과제 3 시드 늘리기 · `statistics.mean` <sub>수업 코드 '과제 3' 빈칸 2곳</sub>

> 🎯 **시드만 바꿔 다섯 번 돌리면 정확도는 얼마나 흔들리며, 모델 차이는 그 흔들림보다 큰가?**

💡 **쉽게 말하면**

- 시드는 **주사위**.
- 가중치 시작값과 배치 순서가 바뀌어 같은 모델도 점수가 달라짐 — 모델 차이가 주사위 차이보다 커야 의미가 있음.

<img src="images/rnn1_seeds.png" alt="seed 5개 흔들림" width="686">

**실행 흐름**

1. 빈칸 `___, ___` · `42, 2026`
2. 세 모델 × 시드 5개
3. 평균 · 폭(최대 - 최소)

**먼저 예측** · RNN · LSTM · GRU 중 시드에 따라 가장 많이 흔들리는 것은? — 답은 코드 실행 결과 아래에

In [22]:
# 과제 3 빈칸 정답: 시드 두 개 추가 (42 · 2026)
# 왜 · seed를 늘려 결과의 흔들림을 더 정확히 재기 위해
SEEDS5 = (SEED, 7, 1234, 42, 2026)                                # 앞 3개 + 빈칸 2개
accs5 = {}                                                        # 모델별 5개 정확도
for kind in ["RNN", "LSTM", "GRU"]:                               # 세 모델
    accs5[kind] = [run_har(kind, 64, seed=s)[0] for s in SEEDS5]  # seed만 바꿔 5번
    a = accs5[kind]                                               # 짧은 이름
    print(f"{kind:<5} {a} · 평균 {statistics.mean(a):.4f}"        # 평균 · 폭
          f" · 폭 {max(a) - min(a):.4f}")

RNN   [0.8073, 0.8246, 0.8039, 0.7465, 0.8113] · 평균 0.7987 · 폭 0.0781


LSTM  [0.8683, 0.889, 0.8748, 0.8833, 0.8928] · 평균 0.8816 · 폭 0.0245


GRU   [0.9026, 0.9006, 0.9002, 0.907, 0.9009] · 평균 0.9023 · 폭 0.0068


**결과** · 예측 정답은 RNN — 폭 `0.0781` (시드 42에서 74.7%)

- `RNN   [0.8073, 0.8246, 0.8039, 0.7465, 0.8113] · 평균 0.7987 · 폭 0.0781` — 시드에 따라 7.8%p 흔들림
- `LSTM  [0.8683, 0.889, 0.8748, 0.8833, 0.8928] · 평균 0.8816 · 폭 0.0245` — 첫 seed가 가장 낮았음 → 6장의 한 번 실행 값은 운이 섞임
- `GRU   [0.9026, 0.9006, 0.9002, 0.907, 0.9009] · 평균 0.9023 · 폭 0.0068` — 가장 안정적

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 번 실행</td><td>seed 운이 섞인 값</td><td>코드가 돌아가는지 빠른 확인</td></tr>
<tr><td>여러 seed 평균 · 폭</td><td>흔들림 크기까지 보임</td><td>모델 비교 결론을 낼 때 (<strong>기본 선택</strong>)</td></tr>
<tr><td>차이 &gt; 흔들림 폭</td><td>구조 차이가 운보다 큼</td><td>"더 낫다"고 말할 수 있음</td></tr>
<tr><td>차이 ≤ 흔들림 폭</td><td>운과 구분 안 됨</td><td>비슷하다고 보고 <strong>비용</strong>(파라미터 · 시간)으로 고름</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한 번 돌린 숫자로 'A가 B보다 낫다'고 쓰지 않기
> - 평균과 폭을 함께 적기

> ✅ **한 줄 정리** · RNN과 LSTM · GRU의 차이(약 8~10%p)는 흔들림보다 크지만, LSTM과 GRU의 차이(2.07%p)는 LSTM의 흔들림 폭(2.45%p) 안이다.

**딥러닝 코드에서는**

- 보고서 표기 예 · `GRU 0.902 ± 0.003 (5 seeds)`
- 평균 ± 표준편차 또는 최소~최대

### 7-3. `A` 과제 4 순서 섞기 검사 · `torch.randperm` <sub>수업 코드 '한 줄 안의 시간 순서를 섞으면' · '과제 4' 빈칸 1곳</sub>

> 🎯 **128칸의 순서를 섞어도 정확도가 거의 그대로라면, 이 데이터에서 순서는 중요한가?**

💡 **쉽게 말하면**

- 순서 섞기는 **문장 속 단어 순서만 뒤섞어 보기**.
- 뜻이 그대로 통하면 그 과제는 단어 순서보다 어떤 단어가 있는지로 풀리는 것.

**실행 흐름**

1. 전부 섞기 함수
2. 빈칸 `___` · `X[b, :half][torch.randperm(half, generator=g)]`
3. LSTM으로 그대로 · 앞 절반 · 전부 섞음 (시드 3개 평균)

**먼저 예측** · 앞 절반만 섞으면 정확도는 '그대로'와 '전부 섞음' 사이에 올까? — 답은 코드 실행 결과 아래에

In [23]:
# 한 줄 안의 128칸 순서를 전부 섞기 (수업 코드)
# 왜 · 모델이 실제로 시간 순서를 쓰고 있는지 섞어서 확인하기 위해
def shuffle_time(X, seed):                                      # 한 줄 안 128칸 섞기
    g = torch.Generator().manual_seed(seed)                     # 이 작업 전용 난수
    out = X.clone()                                             # 원본 보존
    for b in range(X.shape[0]):                                 # 줄마다
        out[b] = X[b][torch.randperm(X.shape[1], generator=g)]  # 칸 순서만 바꿈
    return out                                                  # 섞은 텐서


# 과제 4 빈칸 정답: 앞 64칸만 섞기
# 왜 · 어느 구간의 순서가 중요한지 부분만 섞어 확인하기 위해
def shuffle_front_half(X, seed):                                        # 앞 절반만 섞기
    g = torch.Generator().manual_seed(seed)                             # 이 작업 전용 난수
    out = X.clone()                                                     # 원본 보존
    half = X.shape[1] // 2                                              # 64칸
    for b in range(X.shape[0]):                                         # 줄마다
        out[b, :half] = X[b, :half][torch.randperm(half, generator=g)]  # 빈칸 정답
    return out                                                          # 섞은 텐서


# LSTM hidden 64 · 시드 3개 평균으로 비교 (그대로는 7-2 앞 3개 재사용)
# 왜 · 섞는 범위에 따른 차이가 seed 운이 아닌지 평균으로 비교하기 위해
seeds3 = (SEED, 7, 1234)                                             # seed 3개
normal = accs5["LSTM"][:3]                                           # 그대로는 7-2 결과 재사용
Xs, Xs_te = shuffle_time(Xn, 1), shuffle_time(Xn_te, 2)              # 전부 섞은 데이터
shuffled = [run_har("LSTM", 64, s, Xs, Xs_te)[0] for s in seeds3]    # 전부 섞음 정확도
Xf, Xf_te = shuffle_front_half(Xn, 1), shuffle_front_half(Xn_te, 2)  # 앞 절반 섞은 데이터
front = [run_har("LSTM", 64, s, Xf, Xf_te)[0] for s in seeds3]       # 앞 절반 섞음 정확도
for name, a in [("그대로", normal), ("앞 절반 섞음", front),         # 세 경우
                ("전부 섞음", shuffled)]:
    print(f"{kpad(name, 12)} {a} · 평균 {statistics.mean(a):.4f}")   # 평균 비교

그대로       [0.8683, 0.889, 0.8748] · 평균 0.8774
앞 절반 섞음 [0.8833, 0.8884, 0.8918] · 평균 0.8878
전부 섞음    [0.8751, 0.8673, 0.8738] · 평균 0.8721


**결과** · 예측 정답은 예측과 다름 — 앞 절반 섞음 `0.8878`이 그대로 `0.8774`보다 오히려 높음 (셋 다 흔들림 범위 안)

- `그대로     [0.8683, 0.889, 0.8748] · 평균 0.8774` — LSTM hidden 64 (7-2 앞 시드 3개)
- `앞 절반 섞음 [0.8833, 0.8884, 0.8918] · 평균 0.8878` — 앞 64칸을 섞었는데 1.0%p 높음
- `전부 섞음   [0.8751, 0.8673, 0.8738] · 평균 0.8721` — 전부 섞어도 0.5%p 차이 → LSTM 흔들림 폭(2.45%p)보다 작음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 줄 안 전부 섞기</td><td>값 모음은 그대로, 순서만 파괴</td><td>순서가 중요한지 검사</td></tr>
<tr><td>앞 절반만 섞기</td><td>뒤쪽 순서는 유지</td><td>어느 구간의 순서가 중요한지 검사</td></tr>
<tr><td>줄끼리 섞기 (<code>randperm</code> · <code>shuffle=True</code>)</td><td>줄 안 순서는 그대로, 줄 순서만 바뀜</td><td>학습 배치 만들 때 <strong>항상</strong> (검사가 아님)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 결과로 'HAR에서 순서는 쓸모없다'고 결론 내지 않기
> - 차이가 흔들림 안이면 '이 설정에서는 순서 효과가 드러나지 않았다'까지만

> ✅ **한 줄 정리** · 순서 섞기 검사의 차이가 시드 흔들림보다 작으면, 이 과제는 순서보다 값의 분포(세기 · 흔들림 크기)로 대부분 풀린다고 본다.

**딥러닝 코드에서는**

- 강의 화면(CPU 실행)은 3.59%p 떨어졌고 이 노트북(GPU 실행)은 0.5%p
- 같은 검사도 환경
- seed에 따라 달라지므로 여러 seed로 판단

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 요약 기준선</td><td>합 <code>0.4880</code> · 마지막 값 <code>0.6065</code>, 마지막 칸이 3 · 7인 비율 <code>0.211</code>로 약 60%를 계산</td><td>1-1 · 2-2</td></tr>
<tr><td>2. hidden state</td><td>손으로 편 <code>for</code> 문 = <code>nn.RNN</code> (차이 <code>0.0</code>), 길이 10 · 50 · 200 모두 1,474개</td><td>2-1</td></tr>
<tr><td>3. 모양 · 진짜 끝</td><td>층 2 · 양방향 <code>h_n</code><code>(4, 6, 4)</code>를 손으로, 길이 5~60에서 <code>-1</code> 0.500 vs <code>lens - 1</code> 1.000</td><td>3-3 · 3-4</td></tr>
<tr><td>4. 소실 · 문 · 파라미터</td><td>20칸에서 10만 배 넘게 작아짐, forget bias 2면 99칸 <code>1.6e-05</code>, 1 · 3 · 4벌 손계산</td><td>4장</td></tr>
<tr><td>5. 비교 실험</td><td>GRU <code>0.9026</code> · LSTM <code>0.8683</code> · RNN <code>0.8073</code>, seed 폭 <code>0.0068</code> ~ <code>0.0781</code></td><td>6 · 7장</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>순서를 버리는 요약</td><td>합 · 평균은 순서를 못 봄, 기준선으로 먼저</td><td><code>baseline_acc</code> 표에 함께</td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>hidden state</td><td>매 칸 덮어쓰는 고정 크기 요약, 같은 가중치 반복</td><td><code>nn.RNN(input, hidden)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>세 겹 입력</td><td><code>(배치, 길이, 입력)</code> · 범주는 one-hot</td><td><code>F.one_hot(x, n).float()</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>batch_first</td><td>기본 False, 항상 True로</td><td><code>nn.LSTM(..., batch_first=True)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>out · h_n</td><td>out <code>(N, L, d·H)</code> · h_n <code>(층·d, N, H)</code></td><td><code>self.fc(h_n[-1])</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>마지막 시점</td><td>길이가 다르면 <code>lens - 1</code> · mask</td><td><code>out[idx, lens - 1]</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>기울기 소실</td><td>비율이 반복 곱해져 사라짐, 폭발은 clipping</td><td><code>clip_grad_norm_(..., 5.0)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>LSTM</td><td><code>output, (h_n, c_n)</code>, 문이 열려야 멀리 감</td><td><code>out, (h, c) = self.lstm(x)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>GRU · 파라미터</td><td>1 · 3 · 4벌, 같은 hidden이면 GRU = LSTM × 3/4</td><td><code>sum(p.numel() ...)</code></td></tr>
<tr><td style="text-align: center;"><code>A</code></td><td>정규화</td><td>train 통계로만</td><td><code>(X - mu_train) / sd_train</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;"><code>A</code></td><td>비교 실험</td><td>하나만 바꾸기 · 크기 맞추기 · seed 여러 개</td><td>평균 ± 흔들림</td></tr>
<tr><td style="text-align: center;"><code>B</code></td><td>펼치기 · 사람이 정한 상태</td><td>길이에 묶임 · 규칙을 알 때만</td><td>—</td></tr>
<tr><td style="text-align: center;"><code>B</code></td><td>hidden 크기 · seed</td><td>키운 효과 &lt; 흔들림이면 무의미</td><td><code>for seed in seeds:</code></td></tr>
<tr><td style="text-align: center;"><code>B</code></td><td>순서 섞기 검사</td><td>차이가 흔들림 안이면 순서 효과 불명</td><td><code>X[:, torch.randperm(L)]</code></td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 합 · 평균으로 '3이 7보다 먼저인가'를 풀면 왜 찍기 수준인가?
- **1-2** · 줄 전체를 펼쳐 넣는 방법은 길이가 길어지면 왜 무너지는가?
- **2-1** · hidden state는 무엇이며, 길이가 달라도 파라미터 수가 같은 이유는?
- **3-1** · RNN 입력의 세 겹은 각각 무엇인가?
- **3-2** · `batch_first=False`로 `(배치, 길이, 입력)`을 넣으면 어떻게 되는가?
- **3-3** · 층 2 · 양방향 RNN의 `out` · `h_n` 모양 규칙은?
- **3-4** · 길이가 다른 줄에서 `out[:, -1]`을 읽으면 왜 틀리는가?
- **4-1** · 먼 칸의 기울기는 왜 사라지는가?
- **4-2** · LSTM의 `h_n`과 `c_n`은 무엇이 다르며, 무엇이 기울기를 살리는가?
- **4-3** · 입력 9 · hidden 64에서 RNN · GRU · LSTM 파라미터 수는?
- **5-1** · 정규화 통계는 왜 train에서만 계산하는가?
- **6** · RNN · LSTM · GRU를 공정하게 비교하는 두 가지 방법은?
- **7-2** · 두 모델의 정확도 차이를 믿으려면 무엇과 비교해야 하는가?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

학습 없이 모양 · 파라미터만 확인하는 문제. **먼저 손으로 답을 적고** 아래 빈 셀에서 코드로 확인.

1. `nn.LSTM(input_size=9, hidden_size=16, num_layers=2, batch_first=True)`에 `(5, 128, 9)`를 넣을 때 `out` · `h_n` · `c_n` 모양
2. `batch_first`를 빠뜨린 `nn.GRU(9, 16)`에 같은 `(5, 128, 9)`를 넣으면 `h_n` 모양은? (함정)
3. 숫자 줄 `[[4, 2], [9, 3, 1]]`을 one-hot + 빈 칸 0으로 `(2, 3, 10)` 텐서로 만들고, `nn.RNN(10, 8, batch_first=True)` 출력에서 줄마다 **진짜 마지막 칸**을 꺼내기
4. 입력 9 · hidden 64인 `nn.RNN` · `nn.GRU` · `nn.LSTM` 파라미터 수를 손계산하고 코드로 확인
5. forget 문이 0.5 · 0.88로 100칸 동안 그대로일 때 남는 비율 (`0.5 ** 100`, `0.88 ** 100`)

In [24]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `nn.GRU(9, 16, num_layers=3, bidirectional=True, batch_first=True)`에 `(4, 50, 9)`를 넣으면 `out` · `h_n` 모양은? 손으로 먼저
2. 입력 5 · hidden 20인 `nn.LSTM` 한 층의 파라미터 수를 손계산하고 코드로 확인
3. 길이가 5 · 2 · 4인 세 줄을 5칸 틀로 채우면 `out[:, -1]`이 진짜 끝인 줄은 몇 번째인가?

In [25]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 합 · 평균은 순서를 보지 않는 계산이라 '어느 쪽이 먼저'가 입력에 남지 않음. 모델을 키워도 없는 정보는 안 생김
- **1-2** · 길이마다 입력 · 파라미터가 늘어 같은 데이터로 배우기 어렵고, 학습 때보다 긴 줄은 넣을 수 없음
- **2-1** · 매 칸 덮어쓰는 고정 크기 요약. 모든 칸이 같은 가중치 상자를 반복해 쓰므로(파라미터 공유)
- **3-1** · 배치(몇 줄) · 길이(한 줄에 몇 칸) · 입력 크기(칸 하나를 숫자 몇 개로)
- **3-2** · 에러 없이 배치를 길이로, 길이를 배치로 바꿔 읽음 → `h_n` 가운데가 길이 값
- **3-3** · `out` = (배치, 길이, 방향 × H), `h_n` = (층 × 방향, 배치, H) → `(N, L, 2H)` · `(4, N, H)`
- **3-4** · `-1`은 틀의 끝이라 빈 칸을 지난 요약을 읽음. `out[idx, lens - 1]` 또는 mask 평균
- **4-1** · 칸마다 비슷한 비율(1보다 작음)이 반복 곱해져 지수적으로 작아짐
- **4-2** · `h_n` = 꺼낸 말(출력), `c_n` = 메모. forget 문이 열려 있을 때 메모의 덧셈 길로 신호가 곧게 감
- **4-3** · 4,800 · 14,400 · 19,200 (1 · 3 · 4벌 × (64 × 73 + 128))
- **5-1** · test 정보가 정규화에 새면 평가가 실제보다 좋게 나옴. 추론 때도 train 값을 씀
- **6** · hidden을 같게(구조만) · 파라미터를 같게(크기까지) 두 번 재서 같은 모양인지 확인
- **7-2** · 여러 seed로 잰 흔들림 폭. 차이가 폭보다 작으면 비용으로 고름

#### 코드 문제 답

```python
# 1. out (N, L, H) · h_n · c_n (층, N, H)
lstm = nn.LSTM(input_size=9, hidden_size=16, num_layers=2, batch_first=True)
out, (h_n, c_n) = lstm(torch.randn(5, 128, 9))
shapes = [tuple(t.shape) for t in (out, h_n, c_n)]
print(*shapes)  # (5, 128, 16) (2, 5, 16) (2, 5, 16)

# 2. batch_first=False → 5를 길이로, 128을 배치로 읽음
out, h_n = nn.GRU(9, 16)(torch.randn(5, 128, 9))
print(tuple(h_n.shape))  # (1, 128, 16)

# 3. one-hot + 빈 칸 0 → 줄마다 lens - 1 칸 꺼내기
seqs = [[4, 2], [9, 3, 1]]
lens = torch.tensor([len(s) for s in seqs])
X = torch.zeros(2, 3, 10)
for b, s in enumerate(seqs):
    X[b, :len(s)] = F.one_hot(torch.tensor(s), 10).float()
print(tuple(X.shape), lens.tolist())  # (2, 3, 10) [2, 3]
out, _ = nn.RNN(10, 8, batch_first=True)(X)
last = out[torch.arange(2), lens - 1]
print(tuple(last.shape), torch.equal(last[0], out[0, 1]))  # (2, 8) True

# 4. k벌 × (H × (I + H) + 2H)
hand = [k * (64 * (9 + 64) + 2 * 64) for k in (1, 3, 4)]
code = [sum(p.numel() for p in getattr(nn, m)(9, 64).parameters())
        for m in ("RNN", "GRU", "LSTM")]
print(hand, code)  # [4800, 14400, 19200] [4800, 14400, 19200]
```

```python
# 5. 1보다 작은 수를 100번 곱하기
print(f"{0.5 ** 100:.1e} {0.88 ** 100:.1e}")  # 7.9e-31 2.8e-06
```

#### 변형 문제 답

*문제 → 답*

- **1** · `out` `(4, 50, 32)` (방향 2 × 16) · `h_n` `(6, 4, 16)` (층 3 × 방향 2)
- **2** · 4벌 × (20 × (5 + 20) + 2 × 20) = 4 × 540 = 2160
- **3** · 첫째 줄 하나 (길이 5 = 틀 5칸)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) <code>h_n</code> 모양을 <code>(배치, 층, H)</code>로 답함</td><td><code>batch_first</code>가 <code>h_n</code>에도 적용된다고 착각</td><td><code>h_n</code>은 항상 <code>(층 × 방향, 배치, H)</code></td><td><code>h_n</code>도 <code>batch_first</code>를 따르나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 순서 데이터에 MLP 대신 RNN을 쓰는 이유는?**
> - MLP에 요약값을 넣으면 합
> - 평균처럼 순서를 버리게 되고, 줄 전체를 펼쳐 넣으면 길이마다 입력과 파라미터가 늘어 길어질수록 배우기 어렵고 길이도 고정됩니다.
> - RNN은 고정 크기의 hidden state를 같은 가중치로 칸마다 갱신해, 넘길 정보를 학습으로 정하면서 길이와 무관하게 같은 파라미터를 씁니다.

**Q2. `output`과 `h_n`은 무엇이 다르고, 분류에는 무엇을 쓰나요?**
> - `output`은 마지막 층이 모든 시점에 낸 hidden state로 `(배치, 길이, 방향 × H)`이고, `h_n`은 층과 방향마다 마지막 시점의 hidden state로 `(층 × 방향, 배치, H)`입니다.
> - 한 줄에 라벨이 하나면 마지막 시점을 쓰는데, 길이가 다른 배치에서는 `out[:, -1]`이 패딩을 지난 값이므로 실제 길이로 꺼내거나 `pack_padded_sequence`를 씁니다.
> - 시점마다 라벨이 있으면 `output` 전체를 씁니다.

**Q3. RNN의 기울기 소실은 왜 생기고, LSTM은 어떻게 완화하나요?**
> - 역전파가 시간을 거슬러 갈 때 같은 가중치와 tanh 미분이 반복해서 곱해져, 그 비율이 1보다 작으면 지수적으로 작아집니다.
> - LSTM은 cell state를 곱하기와 더하기로만 갱신하고 forget gate로 남길 양을 정해, 게이트가 열려 있으면 기울기가 멀리까지 전달됩니다.
> - 다만 해결이라기보다 크게 늦추는 것이고, forget bias를 0으로 두면 RNN처럼 사라지는 것을 직접 확인했습니다.

**Q4. LSTM과 GRU 중 무엇을 고르나요?**
> - GRU는 게이트 2개에 cell state가 없어 같은 hidden 크기에서 파라미터가 LSTM의 3/4입니다.
> - UCI HAR에서 hidden을 같게, 또 파라미터를 같게 두 번 비교했고, 여러 seed로 재면 두 모델 차이가 흔들림과 비슷한 크기였습니다.
> - 성능이 비슷하면 파라미터 수와 학습 시간이 적은 GRU부터 시도합니다.

**Q5. 모델 비교 실험에서 주의하는 점은?**
> - 한 번에 하나만 바꾸고, hidden 크기가 아니라 파라미터 수를 맞춘 비교도 함께 합니다.
> - 또 seed를 여러 개 돌려 평균과 흔들림 폭을 보고, 차이가 폭보다 작으면 우열을 주장하지 않습니다.
> - 합이나 평균 같은 기준선도 같이 적어 모델이 실제로 순서를 배웠는지 확인합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>sum()</code> · <code>len()</code> · <code>sorted()</code></td><td>Python 내장 함수</td><td>합 · 길이 · 정렬 (순서를 버리는 요약)</td><td>1-1 <code>v = [sum(s) for s in xs]</code> · <code>sorted(a) == sorted(b)</code></td></tr>
<tr><td><code>nn.Sequential</code> · <code>nn.Linear</code> · <code>nn.ReLU</code></td><td>torch.nn</td><td>MLP 분류기</td><td>1-1 <code>self.net = nn.Sequential(*layers, nn.Linear(width,…</code></td></tr>
<tr><td><code>torch.tensor(...).unsqueeze(1)</code></td><td>torch · Tensor 메서드</td><td>값 목록 → <code>(N, 1)</code> 입력</td><td>1-1 <code>return torch.tensor(v, dtype=torch.float32).unsque…</code></td></tr>
<tr><td><code>.reshape(len(xs), -1)</code></td><td>Tensor 메서드</td><td>줄 전체를 한 줄로 펼치기</td><td>1-2 <code>return to_tensor(xs).reshape(len(xs), -1)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>list.index()</code> · <code>in</code></td><td>list · Python 기본 문법</td><td>먼저 나온 위치 · 포함 여부</td><td>1-2 <code>s.index(3) &lt; s.index(7)</code> · <code>float(3 in s)</code></td></tr>
<tr><td><code>nn.RNNCell</code></td><td>torch.nn</td><td>한 칸 계산 (상자 하나)</td><td>2-1 <code>cell = nn.RNNCell(10, 32)</code></td></tr>
<tr><td><code>nn.RNN</code></td><td>torch.nn</td><td>전체 길이를 안에서 반복</td><td>2-1 <code>rnn = nn.RNN(10, 32, batch_first=True)</code></td></tr>
<tr><td><code>weight_ih_l0</code> · <code>weight_hh_l0</code></td><td>nn.RNN 속성</td><td>입력 · 이전 요약에 곱하는 가중치 (층 0)</td><td>2-1 <code>cell.weight_ih.copy_(rnn.weight_ih_l0)</code></td></tr>
<tr><td><code>torch.stack(hs, dim=1)</code></td><td>torch</td><td>칸별 요약 → <code>(N, L, H)</code></td><td>2-1 <code>manual = torch.stack(hs, dim=1)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>.copy_()</code> · <code>torch.no_grad()</code></td><td>Tensor 메서드 · torch</td><td>가중치 복사 · 기울기 기록 끄기</td><td>2-1 <code>cell.weight_ih.copy_(rnn.weight_ih_l0)</code></td></tr>
<tr><td><code>numel()</code></td><td>Tensor 메서드</td><td>원소 개수 (파라미터 세기)</td><td>0 <code>sum(p.numel() for p in model.parameters())</code></td></tr>
<tr><td><code>F.one_hot(x, num_classes)</code></td><td>torch.nn.functional</td><td>번호 → one-hot 벡터</td><td>3-1 <code>x = F.one_hot(digits, num_classes=10).float()</code></td></tr>
<tr><td><code>batch_first=True</code></td><td>nn.RNN · LSTM · GRU 인자</td><td><code>(배치, 길이, 입력)</code> 순서</td><td>3-2 <code>nn.RNN(10, 32, batch_first=bf)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>num_layers</code> · <code>bidirectional</code></td><td>nn.RNN · LSTM · GRU 인자</td><td>층 쌓기 · 양방향</td><td>3-3 <code>num_layers=layers, bidirectional=bi</code></td></tr>
<tr><td><code>.transpose(0, 1)</code></td><td>Tensor 메서드</td><td>앞 두 축 바꾸기</td><td>3-2 <code>x_bf = x.transpose(0, 1)</code></td></tr>
<tr><td><code>out[:, -1]</code> · <code>out[idx, lens - 1]</code> · <code>h_n[-1]</code></td><td>인덱싱</td><td>마지막 시점 꺼내기</td><td>3-4 <code>output[idx, lens - 1]</code> · 3-3 <code>out[:, -1]</code> · <code>h_n[-1]</code></td></tr>
<tr><td><code>torch.allclose()</code> · <code>torch.equal()</code></td><td>torch</td><td>거의 같은가 · 완전히 같은가</td><td>3-3 <code>torch.allclose(out[:, -1], h_n[-1])</code> · 4-2 <code>torch.equal(h_n, c_n)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>F.cross_entropy</code> · <code>nn.utils.clip_grad_norm_</code></td><td>torch.nn</td><td>분류 손실 · 기울기 자르기</td><td>3-4 <code>F.cross_entropy(model(Xa[j], La[j]), Ya[j]).backwa…</code></td></tr>
<tr><td><code>h.retain_grad()</code> · <code>.grad.norm()</code></td><td>Tensor 메서드</td><td>중간 값 기울기 남기기 · 크기</td><td>4-1 <code>h.retain_grad()</code></td></tr>
<tr><td><code>.to(torch.float64)</code></td><td>Tensor · Module 메서드</td><td>더 정밀한 자료형으로 재기</td><td>4-1 <code>grad_profile(..., dtype=torch.float64)</code> · <code>.to(dtype)</code></td></tr>
<tr><td><code>nn.LSTM</code> · <code>nn.LSTMCell</code></td><td>torch.nn</td><td>문 3개 + 메모 <code>c</code></td><td>4-2 <code>nn.LSTM(10, 32, batch_first=True)</code> · 4-1 <code>nn.LSTMCell</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>output, (h_n, c_n)</code></td><td>Python 튜플 풀기</td><td>LSTM 반환값 받기</td><td>4-2 <code>output, (h_n, c_n) = lstm(x)</code></td></tr>
<tr><td><code>nn.GRU</code></td><td>torch.nn</td><td>문 2개 · 메모 없음</td><td>4-3 <code>gru = nn.GRU(10, 32, batch_first=True)</code></td></tr>
<tr><td><code>torch.sigmoid()</code></td><td>torch</td><td>0~1 문 값</td><td>4-2 <code>gate = torch.sigmoid(torch.tensor([0.0, 2.0, 4.0]))</code></td></tr>
<tr><td><code>urllib.request.urlretrieve()</code> · <code>zipfile.ZipFile()</code></td><td>urllib · zipfile</td><td>내려받기 · 압축 풀기</td><td>5-1 <code>urllib.request.urlretrieve(URL, os.path.join(DATA,…</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>os.path.join()</code> · <code>os.path.exists()</code> · <code>os.makedirs()</code></td><td>os</td><td>경로 · 존재 확인 · 폴더 만들기</td><td>5-1 <code>ROOT = os.path.join(DATA, "UCI HAR Dataset")</code></td></tr>
<tr><td><code>.permute(1, 2, 0)</code> · <code>.contiguous()</code></td><td>Tensor 메서드</td><td>축 순서 바꾸기 · 메모리 정리</td><td>5-1 <code>X = torch.tensor(chans).permute(1, 2, 0).contiguou…</code></td></tr>
<tr><td><code>torch.bincount()</code></td><td>torch</td><td>라벨별 개수</td><td>5-1 <code>print("train 라벨 0~5 개수:", torch.bincount(Yh).tolis…</code></td></tr>
<tr><td><code>.mean((0, 1), keepdim=True)</code> · <code>.std()</code></td><td>Tensor 메서드</td><td>채널별 평균 · 표준편차</td><td>5-1 <code>mu = Xh.mean((0, 1), keepdim=True)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>statistics.mean()</code> · <code>max() - min()</code></td><td>statistics · Python 내장 함수</td><td>평균 · 흔들림 폭</td><td>7-2 <code>print(f"{kind:&lt;5} {a} · 평균 {statistics.mean(a):.4f}"</code></td></tr>
<tr><td><code>torch.Generator().manual_seed()</code></td><td>torch</td><td>이 작업 전용 난수 (전역 seed와 분리)</td><td>7-3 <code>g = torch.Generator().manual_seed(seed)</code></td></tr>
<tr><td><code>torch.randperm(n, generator=g)</code></td><td>torch</td><td>0 ~ n-1을 섞은 순서</td><td>7-3 <code>out[b] = X[b][torch.randperm(X.shape[1], generator…</code></td></tr>
<tr><td><code>X[b, :half] = ...</code></td><td>인덱싱 대입</td><td>앞 절반만 바꿔 넣기</td><td>7-3 <code>out[b, :half] = X[b, :half][...]</code></td></tr>
</tbody></table>

### 새 데이터를 만나면 묻는 아홉 단계 (강의 마지막 30분 · 과제 5)

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">단계</th><th style="text-align: left;">물을 것</th><th style="text-align: left;">이 노트북에서</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">1</td><td>한 샘플이 무엇인가, 길이는 고정인가 가변인가</td><td>0장 · 3-4</td></tr>
<tr><td style="text-align: center;">2</td><td>순서를 섞으면 답이 달라지는가</td><td>1-1 · 7-3</td></tr>
<tr><td style="text-align: center;">3</td><td>달라진다면 무엇을 기억해야 하는가</td><td>1-2 (봤는가 → 먼저 봤는가)</td></tr>
<tr><td style="text-align: center;">4</td><td>넘길 것을 사람이 정해 보면 어디서 깨지는가</td><td>1-1 · 1-2</td></tr>
<tr><td style="text-align: center;">5</td><td>깨지면 넘길 것을 학습으로 정하게 바꾸기</td><td>2-1 · 2-2</td></tr>
<tr><td style="text-align: center;">6</td><td>입력과 출력의 모양 정하기</td><td>3장</td></tr>
<tr><td style="text-align: center;">7</td><td>거리를 늘려 정말 닿는지 성능으로 확인</td><td>4-1</td></tr>
<tr><td style="text-align: center;">8</td><td>안 닿으면 기억 문제인지 학습 문제인지 나눠 보기</td><td>4-1 (60 epoch · 기울기)</td></tr>
<tr><td style="text-align: center;">9</td><td>문이 달린 구조로 바꾸고 통제된 비교로 확인</td><td>4-2 · 4-3 · 6장 · 7장</td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>기준선(합 · 평균 · 끝값)을 먼저</td><td>RNN 점수만 단독으로 보고하기</td></tr>
<tr><td><code>batch_first=True</code> 직접 적기</td><td>기본값 False를 믿고 <code>(배치, 길이, 입력)</code> 넣기</td></tr>
<tr><td>길이가 다르면 <code>lens</code>를 같이 들고 다니기</td><td>패딩한 배치에서 <code>out[:, -1]</code></td></tr>
<tr><td>막히면 <code>x.shape</code> · <code>out.shape</code> · <code>h_n.shape</code> 출력</td><td><code>output.grad</code>로 칸별 기울기 재기</td></tr>
<tr><td>정규화는 train 통계로</td><td>전체(test 포함) 통계로 정규화</td></tr>
<tr><td>hidden 같게 · 파라미터 같게 두 번 비교</td><td>hidden이 같으니 공정하다고 보기</td></tr>
<tr><td>seed 여러 개 평균 · 폭</td><td>한 번 실행 차이로 순위 매기기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>합 · 평균 기준선 vs RNN</td><td>순서를 버림 vs 순서대로 읽고 요약을 학습</td><td>기준선을 먼저 돌리고 RNN이 넘는지 확인</td></tr>
<tr><td><code>nn.RNNCell</code> vs <code>nn.RNN</code></td><td>한 칸 vs 전체 <code>for</code>를 안에서</td><td>학습은 <code>nn.RNN</code>, 측정은 <code>RNNCell</code></td></tr>
<tr><td><code>unsqueeze(-1)</code> vs <code>one_hot</code></td><td>축만 추가 vs 번호 → 벡터</td><td>범주 번호는 <code>one_hot</code> (많으면 <code>Embedding</code>)</td></tr>
<tr><td><code>batch_first=True</code> vs <code>False</code></td><td>(배치, 길이, 입력) vs (길이, 배치, 입력)</td><td><code>True</code>로 통일, <code>h_n</code>은 항상 배치가 두 번째</td></tr>
<tr><td><code>out</code> vs <code>h_n</code></td><td>모든 칸 vs 마지막 칸 (층마다)</td><td>칸마다 답 → <code>out</code>, 한 줄 답 → 마지막 칸</td></tr>
<tr><td><code>h_n</code> vs <code>c_n</code></td><td>꺼낸 말 vs 메모</td><td>분류에는 <code>h_n</code> (또는 <code>out[:, -1]</code>)</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>out[:, -1]</code> vs <code>out[idx, lens - 1]</code></td><td>틀의 끝 vs 데이터의 끝</td><td>길이가 다르면 <code>lens - 1</code></td></tr>
<tr><td>한 방향 vs 양방향</td><td>앞 → 뒤만 vs 앞뒤 모두</td><td>끝이 아직 없는 실시간이면 한 방향</td></tr>
<tr><td>기울기 소실 vs 폭발</td><td>0으로 작아짐 vs 커져서 <code>nan</code></td><td>폭발은 clipping, 소실은 LSTM · GRU</td></tr>
<tr><td>LSTM vs GRU</td><td>문 3 · 줄 2 · 4벌 vs 문 2 · 줄 1 · 3벌</td><td>성능이 비슷하면 비용으로</td></tr>
<tr><td>hidden 같게 vs 파라미터 같게</td><td>구조만 바꾼 비교 vs 크기까지 맞춘 비교</td><td>둘 다 재서 같은 모양인지 확인</td></tr>
<tr><td>train 통계 vs 전체 통계 정규화</td><td>test 정보 차단 vs 새어 들어감</td><td>train 통계</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개를 새로 풀기 · 3-3 손계산표를 가리고 다시 채우기</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>2-1의 손으로 편 RNN을 직접 다시 짜고 <code>nn.RNN</code>과 같은지 확인</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 다시 풀기 · RNN-2 시작 전 3 · 4장 ✅ 줄만 읽기</td><td>15분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- RNN 수업 3자료를 중복 없이 5개 대단원 · 12개 섹션으로 재구성
  - 강의 '순서가 중요하다' · 수업 코드 `rnn.ipynb` · 레이어 실습의 레이어 모양 부분
- 6장 종합 실습(UCI HAR 세 모델 비교) · 7장 수업 과제 빈칸 정답 · 검증(3개 섹션)을 더함
- 정답 코드 · 실행 결과 · 해설 포함, 사인파 예측은 RNN-2 · 텍스트는 RNN-3에서 다룸

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 예측 정답 · 결과 해설 → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 실행 흐름 · 문법 구조

**실행 환경**

- 해설의 수치는 이 노트북을 **GPU(CUDA) · torch 2.11 · seed 고정**으로 두 번 실행해 같은 값이 나온 것.
- GPU가 없으면 CPU로 실행되며 학습 결과 숫자가 조금 다를 수 있음(강의 화면은 CPU 실행값).
- 5장에서 UCI HAR(약 60MB)를 처음 한 번 내려받음

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">수업 자료</th><th style="text-align: left;">내용</th><th style="text-align: left;">섹션</th></tr>
</thead>
<tbody>
<tr><td>강의 1부 · 수업 코드 L0</td><td>합 · 평균 · 마지막 · 첫 값, 질문을 바꾸면</td><td>1-1</td></tr>
<tr><td>강의 1부 · 수업 코드 L0</td><td>통째로 펼치기 · 데이터 줄이기 · 봤는가 · 먼저 봤는가</td><td>1-2 (<strong>통합</strong>)</td></tr>
<tr><td>강의 2부 · 수업 코드 L1</td><td>손으로 펼친 판 · nn.RNN 과 맞춰 보기 · 파라미터 공유</td><td>2-1</td></tr>
<tr><td>수업 코드 L1 '학습시켜 보기'</td><td>RNN 100%</td><td>2-2</td></tr>
<tr><td>강의 '코드로 옮기면' · 수업 코드 '모양 맞추기'</td><td>one_hot · batch_first</td><td>3-1 · 3-2</td></tr>
<tr><td>레이어 실습 RNN · LSTM · GRU (<code>num_layers=2</code>) · 수업 코드 '정말 같은 값일까' · 강의 Bidirectional</td><td>out · h_n · c_n 모양</td><td>3-3 (<strong>통합</strong>: 손계산 표 → 코드 검증)</td></tr>
<tr><td>강의 padding · 수업 코드 L1b</td><td>그대로 텐서로 · 세 줄 채워 보기 · 빈 칸을 읽으면</td><td>3-4</td></tr>
<tr><td>강의 3부 · 수업 코드 L2 · L3</td><td>거리 늘리기 · 기울기 측정 · 0이 진짜 0일까 · 0.9를 50번</td><td>4-1</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">수업 자료</th><th style="text-align: left;">내용</th><th style="text-align: left;">섹션</th></tr>
</thead>
<tbody>
<tr><td>강의 4부 · 수업 코드 L3b · 레이어 실습 LSTM</td><td>LSTM 반환값 · 문을 열어두면 · 식으로 보면</td><td>4-2</td></tr>
<tr><td>강의 5부 · 수업 코드 L4 · 레이어 실습 GRU</td><td>파라미터 수 맞추기</td><td>4-3 · 6장</td></tr>
<tr><td>수업 코드 'UCI HAR 열어보기' · L4</td><td>HAR 불러오기 · 같은 조건 비교</td><td>5-1 · 6장</td></tr>
<tr><td>수업 코드 과제 2 · 3 · 4 (빈칸)</td><td>hidden 크기 · 시드 · 앞 절반 섞기</td><td>7-1 · 7-2 · 7-3</td></tr>
<tr><td>수업 코드 과제 5 · 강의 마지막 30분</td><td>아홉 단계 질문 순서</td><td>11장 요약표</td></tr>
</tbody></table>

수업 코드의 '미리 돌린 결과' 셀 중 같은 결론을 반복하는 것(모델을 키우면 · 학습시킨 모델의 기울기 · 가중치 행렬)은 **한 번만** 대표 실험으로 다룸.

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: center;">순서</th><th style="text-align: left;">배우는 이유</th><th style="text-align: left;">대단원</th></tr>
</thead>
<tbody>
<tr><td style="text-align: center;">1</td><td>순서를 버리는 요약으로는 못 푸는 문제가 있다</td><td>순서가 중요한 데이터</td></tr>
<tr><td style="text-align: center;">2</td><td>넘길 요약을 학습시키는 모델이 필요하다</td><td>hidden state와 RNN</td></tr>
<tr><td style="text-align: center;">3</td><td>그 모델에 넣고 꺼내는 모양을 정확히 맞춘다</td><td>입력 · 출력 모양</td></tr>
<tr><td style="text-align: center;">4</td><td>길어지면 신호가 사라지므로 문을 단 구조로 바꾼다</td><td>멀리까지 닿는가</td></tr>
<tr><td style="text-align: center;">5</td><td>만든 문제가 아닌 실제 센서 데이터를 준비한다</td><td>실제 데이터</td></tr>
<tr><td style="text-align: center;">6</td><td>같은 조건에서 세 구조를 비교한다</td><td>종합 실습</td></tr>
<tr><td style="text-align: center;">7</td><td>조건을 바꿔 결론이 흔들리는지 확인한다</td><td>과제 빈칸</td></tr>
</tbody></table>

#### 순서 데이터 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰이는 장</th></tr>
</thead>
<tbody>
<tr><td>① 데이터 준비</td><td>한 샘플 = 한 줄, 길이 고정인가 · 순서가 중요한가</td><td>1장 · 5장 · 7-3</td></tr>
<tr><td>② 텐서로</td><td><code>(배치, 길이, 입력)</code> · one_hot · padding · lens</td><td>3장 · 5장</td></tr>
<tr><td>③ 전처리</td><td>train 통계로 정규화</td><td>5장</td></tr>
<tr><td>④ 모델</td><td><code>nn.RNN</code> · <code>nn.LSTM</code> · <code>nn.GRU</code> + Linear</td><td>2장 · 4장</td></tr>
<tr><td>⑤ 학습</td><td>미니배치 · Adam · clipping</td><td>0장 · 5장</td></tr>
<tr><td>⑥ 평가</td><td>기준선 · 같은 조건 비교 · seed 흔들림</td><td>1장 · 2장 · 6장 · 7장</td></tr>
</tbody></table>

### 전체 코드

핵심 흐름(순서 과제 RNN → 모양 확인 → UCI HAR 세 모델 비교)을 장별 셀로 모음.

- **위에서부터 이 셀들만 실행해도 동작** (데이터 내려받기 포함)
- 각 출력 앞 `[번호]`가 섹션

**실행 흐름**

1. 전체 코드가 쓰는 도구 · seed · 장치 준비

In [26]:
# 왜 · 데이터 폴더 · 파일 경로를 다루고 실행 환경을 설정하기 위해
import os              # 폴더 · 파일 경로
import random          # 파이썬 난수 (데이터 만들기)
import urllib.request  # 데이터 내려받기
import zipfile         # 압축 풀기

import torch                     # 텐서 · 자동 미분
import torch.nn as nn            # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F  # 함수: one_hot · cross_entropy

SEED = 20260912                            # 수업 코드와 같은 seed
torch.backends.cudnn.deterministic = True  # 같은 GPU · 같은 버전에서 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False     # 실행마다 계산 방법이 바뀌지 않게
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 없으면 CPU


# 왜 · 실험마다 같은 출발점에서 시작하도록 난수를 한 번에 다시 고정하기 위해
def set_seed(seed=SEED):     # seed를 한 번에 고정하는 함수
    random.seed(seed)        # 파이썬 random 고정
    torch.manual_seed(seed)  # CPU · GPU 난수 고정


# 왜 · 모델 크기를 같은 방법으로 세어 비교하기 위해
def n_params(model):                                   # 파라미터 개수 세기
    return sum(p.numel() for p in model.parameters())  # 파라미터 개수

**실행 흐름**

1. 순서 과제 데이터 · 텐서 변환 함수

In [27]:
# ---------- 1 · 2. 순서 과제 → RNN 분류 ----------
# 왜 · 순서 과제 · 기준선 · RNN 분류를 다시 실행하기 위해
# 왜 · 길이 · seed만 바꿔 같은 과제를 여러 번 만들기 위해
def make_order_task(n, length=10, seed=None):                         # 3 · 7 순서 과제 줄 n개
    rng = random.Random(seed)  # 이 데이터 전용 난수 (전역 seed와 분리)
    others = [d for d in range(10) if d not in (3, 7)]                # 3 · 7을 뺀 숫자
    xs, ys = [], []                                                   # 입력 줄 · 정답을 모을 목록
    for _ in range(n):                                                # 줄 n개 만들기
        s = [rng.choice(others) for _ in range(length - 2)]           # 채울 숫자
        i, j = sorted(rng.sample(range(length), 2))                   # 3 · 7 자리
        y = rng.randint(0, 1)                                         # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                           # 정답 1이면 3이 앞
        s.insert(i, a)                                                # 앞 자리에 첫 숫자
        s.insert(j, b)                                                # 뒤 자리에 둘째 숫자
        xs.append(s)                                                  # 완성한 줄 저장
        ys.append(y)                                                  # 정답 저장
    return F.one_hot(torch.tensor(xs), 10).float(), torch.tensor(ys)  # one-hot 입력 · 정답


class Net(nn.Module):                                                  # RNN 분류 모델
    def __init__(self, kind, n_in, hid, n_out):                        # 셀 · 입력 · hidden · 출력
        super().__init__()                                             # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(n_in, hid, batch_first=True)                    # (배치, 길이, 입력)
        self.o = nn.Linear(hid, n_out)                                 # 요약 → 점수

    def forward(self, x):             # 입력이 지나가는 길
        out, _ = self.r(x)            # (N, L, hid)
        return self.o(out[:, -1, :])  # 마지막 칸 요약 → 분류

**실행 흐름**

1. 공통 학습 함수 정의
2. 기준선 · RNN 학습 비교

In [28]:
# 공통 학습 함수: 미니배치 · Adam · (HAR만) 기울기 자르기
# 왜 · 모든 비교가 같은 학습 절차를 쓰고, 긴 줄에서는 기울기 폭발을 막기 위해
def fit(model, X, Y, Xe, Ye, epochs, lr, bs, clip=None):            # 공통 학습 함수
    model = model.to(DEV)                                           # 모델을 GPU로
    X, Y, Xe, Ye = X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)     # 데이터도 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)               # 가중치를 고칠 Adam
    for _ in range(epochs):                                         # epoch 반복
        perm = torch.randperm(len(X)).to(DEV)                       # 줄 순서 섞기
        for i in range(0, len(X), bs):                              # bs줄씩
            j = perm[i:i + bs]                                      # 이번 배치
            opt.zero_grad()                                         # 지난 기울기 지우기
            F.cross_entropy(model(X[j]), Y[j]).backward()           # 손실 → 기울기
            if clip:                                                # HAR만 자르기
                nn.utils.clip_grad_norm_(model.parameters(), clip)  # 기울기 폭발 방지
            opt.step()                                              # 가중치 한 걸음 수정
    model.eval()                                                    # 평가 모드
    with torch.no_grad():                                           # 평가는 기울기 기록 안 함
        return (model(Xe).argmax(1) == Ye).float().mean().item()    # 맞힌 비율 = 정확도


Xtr, Ytr = make_order_task(6000, seed=1)              # 학습
Xte, Yte = make_order_task(2000, seed=2)              # 평가
set_seed()                                            # 모델마다 같은 시작점
acc = fit(Net("RNN", 10, 32, 2), Xtr, Ytr, Xte, Yte,  # 2-2와 같은 조건
          epochs=30, lr=3e-3, bs=128)
print(f"[2-2] RNN · 3과 7 순서 → {acc:.4f}")          # 결과

[2-2] RNN · 3과 7 순서 → 1.0000


**실행 흐름**

1. 2층 · 양방향 RNN의 out · h_n 모양 확인

In [29]:
# ---------- 3 · 4. shape: out (N, L, 방향×H) · h_n (층×방향, N, H) ----------
# 왜 · 출력 모양 규칙을 다시 확인하기 위해
out, h_n = nn.RNN(10, 4, num_layers=2, batch_first=True)(torch.randn(6, 3, 10))  # 2층 RNN 모양
print("[3-3]", tuple(out.shape), tuple(h_n.shape),                               # out · h_n
      torch.allclose(out[:, -1], h_n[-1]))
out, (h_n, c_n) = nn.LSTM(10, 4, batch_first=True)(torch.randn(6, 3, 10))        # LSTM 셋
print("[4-2]", tuple(h_n.shape), tuple(c_n.shape), torch.equal(h_n, c_n))        # h_n ≠ c_n

[3-3] (6, 3, 4) (2, 6, 4) True
[4-2] (1, 6, 4) (1, 6, 4) False


**실행 흐름**

1. UCI HAR 내려받기(처음 한 번) · 읽기 · 정규화
2. RNN · LSTM · GRU 같은 조건 비교

In [30]:
# ---------- 5 · 6. UCI HAR: RNN · LSTM · GRU 같은 조건 비교 ----------
# 왜 · 실제 데이터에서 셀 비교를 다시 실행하기 위해
URL = ("https://archive.ics.uci.edu/static/public/240/"                          # 수업 코드의 주소
       "human+activity+recognition+using+smartphones.zip")
ROOT = os.path.join("UCI_HAR", "UCI HAR Dataset")                                # 압축 푼 폴더
if not os.path.exists(ROOT):                                                     # 처음 한 번만
    os.makedirs("UCI_HAR", exist_ok=True)                                        # 받을 폴더
    urllib.request.urlretrieve(URL, os.path.join("UCI_HAR", "har.zip"))          # 내려받기
    zipfile.ZipFile(os.path.join("UCI_HAR", "har.zip")).extract(                 # 바깥 압축
        "UCI HAR Dataset.zip", "UCI_HAR")
    zipfile.ZipFile(os.path.join("UCI_HAR", "UCI HAR Dataset.zip")).extractall(  # 안쪽 압축
        "UCI_HAR")
CH = [f"{s}_{a}" for s in ("body_acc", "body_gyro", "total_acc")                 # 9채널 이름
      for a in "xyz"]


# 왜 · 학습 · 평가 데이터를 같은 방법으로 읽기 위해
def load_har(split):                                                        # train · test 읽기
    d = os.path.join(ROOT, split)                                           # train 또는 test 폴더
    chans = []                                                              # 채널마다 표
    for c in CH:                                                            # 9채널
        name = f"{c}_{split}.txt"                                           # 파일 이름
        with open(os.path.join(d, "Inertial Signals", name)) as f:          # 채널 파일
            chans.append([[float(v) for v in line.split()] for line in f])  # 줄마다 128개 숫자
    with open(os.path.join(d, f"y_{split}.txt")) as f:                      # 정답 파일
        y = torch.tensor([int(v) - 1 for v in f.read().split()])            # 1~6 → 0~5
    return torch.tensor(chans).permute(1, 2, 0).contiguous(), y             # (N, 128, 9)


# 왜 · 학습 · 평가 데이터를 한 번만 읽어 두고 뒤에서 다시 쓰기 위해
Xh, Yh = load_har("train")                                                   # 학습 7352줄
Xh_te, Yh_te = load_har("test")                                              # 평가 2947줄
mu, sd = Xh.mean((0, 1), keepdim=True), Xh.std((0, 1), keepdim=True)         # train 통계
for kind in ["RNN", "LSTM", "GRU"]:                                          # 세 모델
    set_seed()                                                               # 모델마다 같은 시작점
    model = Net(kind, 9, 64, 6)                                              # 같은 hidden 64
    acc = fit(model, (Xh - mu) / sd, Yh, (Xh_te - mu) / sd, Yh_te,           # 6장과 같은 조건
              epochs=12, lr=1e-3, bs=64, clip=5.0)
    print(f"[6] {kind:<4} 정확도 {acc:.4f} · 파라미터 {n_params(model):,}")  # 결과

[6] RNN  정확도 0.8073 · 파라미터 5,190


[6] LSTM 정확도 0.8683 · 파라미터 19,590


[6] GRU  정확도 0.9026 · 파라미터 14,790
